# EDHEC Risk Institute – Portfolio Management
## Cuaderno Consolidado

Este notebook fusiona los 8 cuadernos del curso en un solo documento ordenado por capítulo.

| Sección | Contenido |
|---|---|
| Chapter 1 | Analysing Returns, VaR & CVaR |
| Chapter 1 Lab | Portfolio Analysis – Analyzing Returns |
| Chapter 2 | Efficient Frontier & Markowitz Portfolio |
| Chapter 2 Lab | Portfolio Optimization |
| Chapter 3 Lab | Limits of Diversification |
| Chapter 3 | Constant Proportion Portfolio Insurance (CPPI) |
| Chapter 4 | Introduction to Asset-Liability Management |
| Chapter 4 Lab | Asset-Liability Management |

---


---

# Chapter 1: Analysing Returns, VaR & CVaR

---


# Chapter 1: Analysing Returns, VaR, and CVaR

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats 
import edhec_risk_kit as erk

%load_ext autoreload
%autoreload 2
%matplotlib inline

# using seaborn style (type plt.style.available to see available styles)
plt.style.use("seaborn-pastel")  

### Returns

Definition of simple (price) **return**: how much we gain (or loose) from the price at time $t+1$ and the price at time $t$. 
That is:

$$
R_{t,t+1} := \frac{P_{t+1} - P_t}{P_{t}} = \frac{P_{t+1}}{P_t} - 1. 
$$


### Compound return

In general, given a time frame $(t, t+k)$, with $k>1$:
$$
R_{t,t+k} = (1+R_{t,t+1})(1+R_{t+1,t+2})\cdots(1+R_{t+k-1,t+k}) - 1.
$$

If the returns are the same over all time periods, say $R$, for the entire given time frame $(t, t+k)$, with $k>1$, 
then the compound returns becomes simply
$$
R_{t,t+k} = (1+R)^{k} - 1.
$$

### Return per year (or annualized return)


**In general**:
$$
R_{py} = (1 + R)^{P_{y}} - 1,
\quad\text{where}\quad
P_{y} =
\begin{cases}
&252  & \text{if $R$ is daily return},\\
&52   & \text{if $R$ is weekly return},\\
&12  & \text{if $R$ is monthly return}
\end{cases}
$$
The variable $P_y$ is called **periods_per_year**.


### Volatility (or risk)

The **volatility** of an asset is simply the standard deviation of the returns of the asset:
$$
\sigma := \sqrt{  \frac{1}{N-1} \sum_{t} (R_t - \mu)^2  },
$$
where $R_t$ is our series of returns at time $t$ and $\mu$ is the sample mean of the returns, i.e., $\mu := \frac{1}{N}\sum_{t}R_t$, 
with $N$ denoting the number of returns.

### Adjusting the volatility

Suppose that we have **monthly** returns and we compute the volatily of our asset, that is, we compute the **monthly volatility**. 
What if we want to know the **volatility over the year?** 
It is clear that we cannot compare the volatility obtained from data corresponding to different time scales. 
The way to proceed is the following:
$$
\sigma_{ann} = \sigma_{p} \sqrt{p},
$$
where $\sigma_{ann}$ is the **annualized volatility**, or volatility per year, whereas $p$ stands for the period considered 
and $\sigma_p$ the corresponding computed volatility. 

For example:

1) In case of **monthly** returns with volatility $\sigma_m$, we compute the annualized volatility 
by doing $\sigma_{ann}=\sigma_m\sqrt{12}$;

2) In case of **weekly** returns with volatility $\sigma_w$, we compute the annualized volatility 
by doing $\sigma_{ann}=\sigma_w\sqrt{52}$;

3) In case of **daily** returns with volatility $\sigma_d$, we compute the annualized volatility 
by doing $\sigma_{ann}=\sigma_d\sqrt{252}$.

### Return on risk 

This is a measure to obtain **how much reward (return) do we get from our the investment in some asset per unit of risk** (volatility). 
By definition, this is simply the ratio between the return and the volatility:
$$
\text{ROR} := \frac{\text{RETURN}}{\text{RISK}} = \frac{R}{\sigma}, 
$$
where RETURN is the total (compound) return over the period under consideration. 
Let us compute the RORs for the two stocks in the example above.

### Sharpe Ratio
The **sharpe ratio** is defined as:
$$
\lambda := \frac{E_R}{\sigma}
\quad\text{where}\quad
E_R := R - R_F, 
$$
where $E_R$ is called the **excess return** which is therefore nothing but that the return $R$ minus a **benchmark (risk-free) return**.

### Example

In [ ]:
ffme = erk.get_ffme_returns()
ffme.head()

In [ ]:
ffme.plot(grid=True, figsize=(10,4))
plt.show()

In [ ]:
# compute the volatility
vol = ffme.std()
vol

Note that this is the **monthly** volatility since we have monthly data. Compute the annualized volatility:

In [ ]:
annualized_vol = vol * np.sqrt(12)
annualized_vol

Now, we want to compute the **return per month**. We need the total numbers of months of the entire timeframe (from 1926 to 2018), which is simply the number of rows of the dataframe. Then we can use the formula for the return per month:

In [ ]:
nmonths = ffme.shape[0]
total_return = (1 + ffme).prod() - 1
return_per_month = (1 + total_return)**(1/nmonths) - 1
return_per_month

Now compute the **return per year** (**annualized return**) by using either the return per month or using the total return.

In [ ]:
# return-per-year: using the formula with return per month and power 12
annualized_return = (1 + return_per_month)**12 - 1
print( annualized_return ) 

# which is the same as:
# return-per-year: using the formula with total return and power 12/no. of months  
#annualized_return = (1 + total_return) ** (12/nmonths) - 1
#print( annualized_return ) 

Compute the **ROR and sharpe ratios**:

In [ ]:
ROR = annualized_return / annualized_vol
ROR

This would suggest to invest in Large caps (Hi 10) due to higher return per unit of risk. However:

In [ ]:
# define a risk free rate
risk_free_rate = 0.03
excess_return  = annualized_return - risk_free_rate
sharpe_ratio   = excess_return / annualized_vol
sharpe_ratio

In in case of a hypothetical risk-free rate of $3\%$, then we get higher sharpe ratio for Small Caps (Lo 10)$\dots$

In [ ]:
excess_return

### Drawdown

The Drawdown is defined as the **worst return** we could have experienced if we entered and left the market at the worst possible points. (e.g. Buy at the highest, Sell at the lowest). 
It measures **potential losses**, and therefore it is a **downside risk measure**.

In order to compute the drawdon of the two indices we do the following steps


From EDHEC's note:

1. Compute the so-called **wealth index**, i.e., the value of the portfolio as it compounds over times. That is, given the series of returns, it is the series of compound returns at each time frame (using *cumprod()* method) 
2. Compute previous peaks
3. Compute the **drawdown**, i.e., the wealth values as a percentage of previous peaks


The **drawdown** is simply the difference of the wealth index from the (last) max peak, that is:
1. wealth_index - previous_peaks  (*in absolute values*)
2. (wealth_index - previous_peaks) / previous_peaks (*in percentage*)

In [ ]:
# we start from $100 and see how they evolve according to the returns
wealth_index = 100 * (1 + ffme).cumprod()
wealth_index.head()

f, ax = plt.subplots(figsize=(20,5), nrows=1, ncols=2)
# Plot of the wealth indices
wealth_index["SmallCap"].plot(grid=True, ax=ax[0], label="small caps", legend=True)
wealth_index["LargeCap"].plot(grid=True, ax=ax[1], label="large caps", legend=True )
# Using the cummax() method we can compute the cumulative max (peaks) throughout the series
previous_peaks = wealth_index.cummax()
previous_peaks["SmallCap"].plot(title="Small Caps with max peaks", grid=True, ax=ax[0], label="max peaks", legend=True)
previous_peaks["LargeCap"].plot(title="Large Caps with max peaks", grid=True, ax=ax[1], label="max peaks", legend=True)
plt.legend()
plt.show()

In [ ]:
f, ax = plt.subplots(2,2,figsize=(20,12))
# Plot of the wealth indices
wealth_index["SmallCap"].plot(grid=True, title="Small Caps", ax=ax[0,0]) 
wealth_index["LargeCap"].plot(grid=True, title="Large Caps", ax=ax[0,1]) 

drawdown = (wealth_index - previous_peaks) / previous_peaks
(drawdown["SmallCap"]*100).plot(grid=True, title="Drawdown Small Caps", ax=ax[1,0], color='r')
(drawdown["LargeCap"]*100).plot(grid=True, title="Drawdown Large Caps", ax=ax[1,1], color='r')
ax[1,0].set_ylabel("%")
ax[1,1].set_ylabel("%")
plt.show()

For example, we see that after the '29 crisis there has been a loss of over $80\%$ of wealth for those people investing in Large Caps:

In [ ]:
print("'29 crisis: ")
print( "{}%" .format( drawdown.min().round(2)*100) )
print("Date max drawdown:")
print( drawdown.idxmin() )

The other two large drawdowns occured during the **dot com** crisis a the beginning of the new century and due to the **Lehman Brothers** crisis:

In [ ]:
print("Dot Com crisis: ")
print( "{}%" .format( drawdown["1990":"2005"].min().round(2)*100) )
print("Date max drawdown:")
print( drawdown["1990":"2005"].idxmin() )

In [ ]:
print("Lehman Brothers crisis: ")
print( "{}%" .format( drawdown["2005":].min().round(2)*100) )
print("Date max drawdown:")
print( drawdown["2005":].idxmin() )

### Skewness and Kurtosis 

The **skewness** is a measure of the asymmetry of the probability distribution of a real-valued random variable 
about its mean. It **can be positive or negative, or undefined**.
For a unimodal distribution, **negative skewness commonly indicates that the fat tail is on the left side of the distribution**, and positive skewness indicates that the fat tail is on the right. 

The formal definition involves the third centered moment:
$$
S(X) := \frac{\mathbb{E}([X - \mu]^3)}{\sigma^3},
$$
where $\sigma$ is the standard deviation of $X$.

The **kurtosis** is a measure of the **tailedness** of the probability distribution of a real-valued random 
variable, that is, it is a descriptor of the shape of a probability distribution. 
The formal definition involves the fourth centered moment:
$$
K(X) := \frac{\mathbb{E}([X - \mu]^4)}{\sigma^4},
$$
where $\sigma$ is the standard deviation of $X$. Basically, Kurtosis is **the average of the standardized data raised to the fourth power**. Any standardized values that are less than 1 (i.e., data within one standard deviation of the mean, which is where we observe the "peak"), contribute virtually nothing to kurtosis, since raising a number that is less than 1 to the fourth power makes it closer to zero. The only data values (observed or observable) that contribute to kurtosis in any meaningful way are those outside the region of the peak; i.e., the outliers. Therefore, **kurtosis measures outliers only**, saying nothing about the peak.


**If $X$ is a Gaussian random variable we have $S(X) = 0$ and $K(X)=3$.**

In particular, the **Excess Kurtosis** is defined as Kustosis minus 3, in order to provide a comparison to the normal distribution.

#### Examples

In [ ]:
# Normal distributed random variable with mean 0 and std 2
A = pd.DataFrame( {"A" : np.random.normal(0, 2, size=800)} )

# Returns from FF dataset, that we know that are NOT normally distributed
B = erk.get_ffme_returns()
B = B["LargeCap"]

f = plt.figure(figsize=(18,3))
ax1 = f.add_subplot(121)
ax2 = f.add_subplot(122)

ax1.hist( A.values ,bins=60, density=True )
ax1.set_title('Normal r.v. - Mean {}; Std {}' .format(A.mean().values.round(3),A.std().values.round(3)))
ax1.grid()

ax2.hist( B.values * 100 ,bins=60, density=True )
ax2.set_title('Not Normal r.v. - Mean {}; Std {}' .format(np.round(B.mean(),3), np.round(B.std(),3) ))
ax2.grid()

From the plot of the distributions, we can see that for the not normal random variables, we have a kind of symmetry, **but tails are fatters**. 
We then expect **the skewness to be close to zero** whereas the **kurtosis to be higher than $3$**.

In [ ]:
# Skewness of A and B
S_A = ( (A - A.mean())**3 / A.std(ddof=0)**3 ).mean() 
K_A = ( (A - A.mean())**4 / A.std(ddof=0)**4 ).mean() 

S_B = ( (B - B.mean())**3 / B.std(ddof=0)**3 ).mean()
K_B = ( (B - B.mean())**4 / B.std(ddof=0)**4 ).mean()

f = plt.figure(figsize=(18,8))
ax1 = f.add_subplot(221)
ax2 = f.add_subplot(222)
ax3 = f.add_subplot(223)
ax4 = f.add_subplot(224)

ax1.plot( ((A - A.mean())**3 / A.std(ddof=0)**3).values  )
ax1.set_title('Normal RV: plot of "skewness before the mean" - S={}' .format(S_A.values) )
ax1.axhline(y=S_A[0], linestyle=":", color="red")
ax1.grid()

ax2.plot( ((A - A.mean())**4 / A.std(ddof=0)**4).values  )
ax2.set_title('Normal RV: plot of "kurtosis before the mean" - K={}' .format(K_A.values) )
ax2.axhline(y=K_A[0], linestyle=":", color="red")
ax2.grid()

ax3.plot( ((B - B.mean())**3 / B.std(ddof=0)**3).values  )
ax3.set_title('Not Normal RV: plot of "skewness before the mean" - S={}' .format(np.round(S_B,3)) )
ax3.axhline(y=S_B, linestyle=":", color="red")
ax3.grid()

ax4.plot( ((B - B.mean())**4 / B.std(ddof=0)**4).values  )
ax4.set_title('Not Normal RV: plot of "kurtosis before the mean" - S={}' .format(np.round(K_B,3)) )
ax4.axhline(y=K_B, linestyle=":", color="red")
ax4.grid()

Let us load another dataset corresponding to **hedge fund indices**:

In [ ]:
hfi = erk.get_hfi_returns()
hfi.head(3)

In [ ]:
hfi_skew_kurt = pd.DataFrame(columns=["Skewness","Kurtosis"])

# Compute the skewness and kurtosis of the returns in hfi using the pandas aggregate method, 
# which takes in input a function and applies the function to every column of the given dataframe

# compute the skewness
hfi_skew_kurt["Skewness"] = hfi.aggregate( erk.skewness )

# compute the kurtosis
hfi_skew_kurt["Kurtosis"] = hfi.aggregate( erk.kurtosis )

hfi_skew_kurt

Which index has, more or less, a Gaussian distribution? It seems that **CTA Global** has a skewnesss close to zero and kurtosis close to 3. 

We can use a test, called **Jarque-Bera test** from **scipy** which is implemented in our erk toolkit.

In [ ]:
# For example:
print( scipy.stats.jarque_bera( hfi["CTA Global"] ) )
print( erk.is_normal( hfi["CTA Global"] ) )

In [ ]:
#while:
print( scipy.stats.jarque_bera( hfi["Convertible Arbitrage"] ) )
print( erk.is_normal(  hfi["Convertible Arbitrage"] ) )

The second value is the so-called **p value** which is the one to look at to see if the returs are normally distributed. 
By default, if this value is larger than $0.01$, then the answer is true, or if you want, the series of returns passes the test.

In [ ]:
hfi.aggregate( erk.is_normal )

We see that only the CTA GLobal index of returns passes the test, i.e., it seems to be normally distributed.

## Downside risk measures

### Semivolatility (or semideviation)

The **semivolatility** is the volatility of the portion of the return dataset which are negative.


**We rather concern about the volatility of negative returns**. Therefore, the definition is given by:
$$
\sigma_{semi} := \sqrt{ \frac{1}{N_{semi}} \sum_{R_t < 0} (R_t - \mu_{semi})^2 },
$$
where $\mu_{semi}$ is the sample mean of the negative returns and $N_{semi}$ is the number of these negative returns. 

Note that the definition can be applied to the returns which are **below the mean**, not necessarily at the negative ones.

In [ ]:
# Computing the semivolatility (for negative returns) implemented in erk
erk.semideviation( hfi )

### Value at Risk (VaR)

It represents the **maximum expected loss** over a certain time period. 
First of all, we specify a certain confidence **level**, in $(0,1)$, although it is typically expressed in percentage. 

For example consider a $99\%$ level (i.e., $\alpha=0.99$). When we say **$99\%$ monthly VaR** it means that we are looking at the **worst possibile outcome over a month after excluding the $1\%$ of extreme worst losses**. In other words, what is **the maximum loss that you can take with $99\%$ of probability  over one month**.

**Example:** we are given the following set of monthly returns:
$$
R = (-4\%, +5\%, +2\%, -7\%, +1\%, +0.5\%, -2\%, -1\%, -2\%, +5\%).
$$
**What is the $90\%$ monthly VaR?** 

So what we have to do is 1) to exclude the $10\%$ worst returns and 2) looking at the worst return of the remaining ones. 
Since we have $10$ returns, the $10\%$ worse return is just $1$ return, i.e., $-7\%$, and so the worse return of the remaing ones is $-4\%$. 
Then $\text{VaR} = 4\%$.

**NOTE THAT although the value we find was $-4\%$ we say that $\text{VaR} = 4\%$, i.e., the VaR is typically a positive number.**


Mathematically, the VaR is defined as follow. Given the confidence level $\alpha\in(0,1)$,
$$
\text{VaR}_{\alpha}
:= - \text{inf}\{x\in\mathbb{R} \;:\; \mathbb{P}(R \leq x) \geq 1-\alpha \} 
= - \text{inf}\left\{x\in\mathbb{R} \;:\;  \mathbb{P}(R \geq x) \leq \alpha\right\},  
$$
that is, it is nothing but that the **$(1-\alpha)$-quantile** since we effectively want to find the number $\text{VaR}_\alpha$ such that 
$$
\mathbb{P}( R \leq -\text{VaR}_\alpha) = 1-\alpha,
$$
which says that there is a $(1-\alpha)\%$ probability of having a (negative) return greater or equal to $-\text{VaR}_\alpha$.

In previous example, the $90\%$ monthly VaR equal to $4\%$, means that 
$$
0.04 = \text{VaR}_{0.9} = -\text{inf}\{x\in\mathbb{R} \;:\; \mathbb{P}(R \leq x) \geq 0.01 \},
$$
i.e., there is a $10\%$ probability of loosing more than $4\%$ of our money invested (having monthly returns less than -$4\%$). 

### Conditional VaR (or Beyond VaR)

This is defined as the expected loss **beyond** VaR, or more formally, we look at **the average of the distribution beyond the VaR**, that is of those returns which are less than the VaR. Mathematically, this is going to be:
$$
\text{CVaR} := - \mathbb{E}( R | R<-\text{VaR}) = - \frac{\int_{-\infty}^{-\text{VaR}} t f_R(t)dt }{F_R(-\text{Var})},
$$
where $f_R$ is the density function of our returns and $F_R$ is the cumulative distribution function. 

**Example:** we are given the following set of monthly returns:
$$
R = (-4\%, +5\%, +2\%, -7\%, +1\%, +0.5\%, -2\%, -1\%, -2\%, +5\%).
$$
**What is the $80\%$ monthly CVaR?** 

So what we have to do is 1) excluding the $20\%$ worse returns and 2) looking at the worse return of the remaining ones and in this way we find $\text{VaR}_{0.8}$. 3) Then, we take the average of the returns which are less than $\text{VaR}_{0.8}$. 

Since we have $10$ returns, the $20\%$ worse return are $2$ returns, $-7\%$ and $-4\%$. The worse return among the remaining ones is $-2\%$. That is, $\text{VaR}_{0.8} = 2\%$. Now, we see that the returns less than $-\text{VaR}_{0.8}=-2\%$ are only $-7\%$ and $-4\%$. 
Their average is $\text{CVAR}_{0.8} = - (-7\%-4\%)/2 = 5.5\%$.

### Methods for estimating VaR and CVaR
We have the following methods:

#### Historical method (non parametric)
This is the most intuitive one which works by simply applying the definition of VaR as $(1-\alpha)$-quantile of the distribution of the returns of the asset under consideration. For example, consider the hedge fund indices returns:

In [ ]:
# get the CTA global returns
hfi = erk.get_hfi_returns()

# ...and plot their distribution
ax = hfi["CTA Global"].plot.hist(figsize=(8,4), bins=60, density=True)
ax.set_title("CTA Global returns distribution")
ax.grid()

We want to get the $90\%$, $95\%$, and $99\%$ monthly VaR. 
That is, we have levels $1-\alpha=0.01, 0.05, 0.01$. We can use the *percentile* method.

In [ ]:
alpha = np.array([0.90, 0.95, 0.99])
level = 1 - alpha

# In the percentile method, we multiply by 100 because it wants an input between 0 and 100
VaRs = -np.percentile(hfi["CTA Global"], level*100)

print("90% Var: {:.2f}%".format(VaRs[0] * 100))
print("95% Var: {:.2f}%".format(VaRs[1] * 100))
print("99% Var: {:.2f}%".format(VaRs[2] * 100))

It means that there is a $10\%$, $5\%$, $1\%$ probability that any given month we can loose at least about $2.4\%$, $3\%$, and $5\%$, respectively. 

Or, alternatively, that **there is a $90\%$, $95\%$, $99\%$ probability that for any given month we loose less than $2.4\%$, $3\%$, and $5\%$, respectively.**

It is worth saying that this way of computing VaR is, however, sensitive to the timescale of our returns, 
because a VaR computed using monthly returns will be different from a VaR computed using weekly returns$\dots$

#### Parametric method (Gaussian)
Here, we **assume that the returns are normally distributed**, which is, however, **often incorrect**. 

Let $\mu$ and $\sigma$ be the mean and the volatility of the returns $R$ and suppose that $R\sim N(\mu,\sigma)$. 
Via standardization, we can express $R$ as $R = \mu + X \sigma$, where $X\sim N(0,1)$. 
This way, **for computing $\text{VaR}_\alpha$, i.e. the $(1-\alpha)$-quantile of the distribution of $R$, we can compute the $(1-\alpha)$-quantile of the stardard normal distribution**. 

By definition of $\text{VaR}_\alpha$ and quantiles, we want to find the number $z_\alpha$ such that 
$$
\mathbb{P}(R \leq z_\alpha) = 1-\alpha.
$$
Hence we would have:
$$
1-\alpha = \mathbb{P}(R \leq z_\alpha) = \mathbb{P}(\mu+ X\sigma \leq z_\alpha) 
= \mathbb{P}\left(X \leq \frac{z_\alpha-\mu}{\sigma}\right) 
= \Phi\left( \frac{z_\alpha-\mu}{\sigma} \right)
\qquad\Longrightarrow\qquad
z_\alpha = \mu + \Phi^{-1}(1-\alpha)\sigma 
$$
Hence, we have found:
$$
\text{VaR}_\alpha = -\left(\;\mu + \Phi^{-1}(1-\alpha) \sigma\;\right),
$$
where $\Phi^{-1}(1-\alpha)$ is the $(1-\alpha)$-quantile of the Gaussian distribution that we can find using the *norm.ppf*, 
and $\mu$ and $\sigma$ are the mean and volatility of our returns series, respectively 
(here, recall that we put a minus since we want the VaR to be a positive number).

In [ ]:
# Compute the 95% monthly Gaussian VaR of the hedge fund indices 
alpha = 0.95
erk.var_gaussian( hfi, level=(1-alpha)*100)

#### Cornish-Fisher method (semi parametric)

This is a modification of the parametric Gaussian method. The method uses the **Cornish-Fisher expansion (1937) of quantiles** which basically relates the $\alpha$-quantilies of **non Gaussian** distribution with the $\alpha$-quantiles of the Gaussian distribution 
in the following way:
$$
\tilde{z}_\alpha 
= z_\alpha + \frac{1}{6}(z_\alpha^2 - 1)S 
+ \frac{1}{24}(z_\alpha^3 - 3 z_\alpha)(K-3) 
- \frac{1}{36}(2z_\alpha^3 - 5 z_\alpha)S^2
$$
where $\tilde{z}_\alpha$, $S$, and $K$ denote the $\alpha$-quantile, the skewness, and the kurtosis of the the non Gaussian distribution (say, our returns series), respectively, and $z_\alpha$ is the $\alpha$-quantile of the Gaussian distribution. 
Notice that if the distribution of our series was, effectively, Gaussian, then $S=0$ and $K=3$ and so $\tilde{z}_\alpha$ would be equal to $z_\alpha$.

Therefore, with this method, we have:
$$
\text{VaR}_\alpha = -\left(\;\mu + \tilde{z}_\alpha  \sigma\;\right).
$$

In [ ]:
# Compute the 95% monthly Gaussian VaR of the hedge fund indices using the Cornish-Fisher method
erk.var_gaussian(hfi, level = (1-alpha)*100, modified=True)

Finally, the **conditional VaR** is computed using the historical method: look at the *cvar_historic* from the *erk* toolkit.

In [ ]:
erk.cvar_historic(hfi)

#### Compare VaRs

In [ ]:
comparevars = pd.concat([erk.var_historic(hfi), erk.var_gaussian(hfi), erk.var_gaussian(hfi,modified=True), erk.cvar_historic(hfi)], axis=1)
comparevars.columns = ["Historical","Gaussian","Cornish-Fisher","Conditional VaR"]
(comparevars * 100).plot.bar(figsize=(13,5), grid=True, title="Comparison of 95% monthly VaRs for Hedge Fund indices")
plt.ylabel("%")
plt.show()


---

# Chapter 1 (Lab): Portfolio Analysis – Analyzing Returns

---


## Analyzing Returns

### Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import edhc_risk_kit as erk
import matplotlib.pyplot as plt
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [ ]:
df = pd.read_csv('data/Portfolios_Formed_on_ME_monthly_EW.csv',
                  index_col=0, na_values=-99.99)
df.index = pd.to_datetime(df.index, format='%Y%m').to_period('M')
df.head()

Recall that the return from time $t$ to time ${t+1}$ is given by:

$$ R_{t,t+1} = \frac{P_{t+1}-P_{t}}{P_{t}} $$

or alternately

$$ R_{t,t+1} = \frac{P_{t+1}}{P_{t}} - 1 $$

### Analysis of the top quintiles from investments in a portfolio.

In [ ]:
returns = df[['Lo 20', 'Hi 20']] / 100
#returns.columns = ['Small Caps', 'Large Caps']
returns.plot.line(figsize=(15,5));

**What was the Annualized Return of the `Lo 20` and `Hi 20` portfolios over the entire period?**

In [ ]:
print(erk.annualize_rets(returns, 12))

**What was the Annualized Volatility of the `Lo 20` and `Hi 20` portfolios over the entire period?**

In [ ]:
print(erk.annualize_vol(returns, 12))

**What was the Annualized Return of the `Lo 20` and `Hi 20` portfolios over the period 1999 - 2015 (both inclusive)?**

In [ ]:
print(erk.annualize_rets(returns['1999':'2015'], 12))

**What was the Annualized Volatility of the `Lo 20` and `Hi 20` portfolios over the period 1999 - 2015 (both inclusive)?**

In [ ]:
print(erk.annualize_vol(returns['1999':'2015'], 12))

**What was the Maximum Drawdown (expressed as a positive number) experienced over the 1999-2015 period in the `Lo 20` and `Hi 20` portfolios?**

In [ ]:
res_lo20 = erk.drawdown(returns['1999':'2015']['Lo 20'])['Drawdown'].min()
print('Maximum drawdown of Lo 20:', round(-res_lo20 * 100,2))

In [ ]:
erk.drawdown(returns['1999':'2015']['Lo 20'])['Drawdown'].idxmin()

In [ ]:
res_hi20 = erk.drawdown(returns['1999':'2015']['Hi 20'])['Drawdown'].min()
print('Maximum drawdown of Hi 20:', round(-res_hi20 * 100,2))

In [ ]:
erk.drawdown(returns['1999':'2015']['Hi 20'])['Drawdown'].idxmin()

In [ ]:
hfi = erk.get_hfi_returns() # Get hedge fund indices
hfi.head()

**What is the skewness of the indices?**

In [ ]:
print(erk.skewness(hfi['2009':]).sort_values())

**What is the kurtosis of the indices?**

In [ ]:
print(erk.kurtosis(hfi['2009':]).sort_values())

## Downside Measures

### Semideviation

Semideviation is an alternative measurement to standard deviation or variance. However, unlike those measures, semideviation looks only at negative price fluctuations. Thus, semideviation is most often used to evaluate the downside risk of an investment.

**What is the semideviation of the indices?**

In [ ]:
print(erk.semideviation(hfi['2009':]).sort_values())

### Value at Risk (VaR) and Conditional Value at Risk (CVaR)

Value at risk (VaR) is a statistic that measures and quantifies the level of financial risk within a firm, portfolio or position over a specific time frame. This metric is most commonly used by investment and commercial banks to determine the extent and occurrence ratio of potential losses in their institutional portfolios. Risk managers use VaR to measure and control the level of risk exposure. One can apply VaR calculations to specific positions or whole portfolios or to measure firm-wide risk exposure.  

Conditional Value at Risk (CVaR), also known as the expected shortfall, is a risk assessment measure that quantifies the amount of tail risk an investment portfolio has. CVaR is derived by taking a weighted average of the “extreme” losses in the tail of the distribution of possible returns, beyond the value at risk (VaR) cutoff point. Conditional value at risk is used in portfolio optimization for effective risk management.

### Historic VaR

The historical method simply re-organizes actual historical returns, putting them in order from worst to best. It then assumes that history will repeat itself, from a risk perspective. The code below shows that for the first index, Convertible Arbitrage, there is a 5% chance you might lose approximately 1.5%, or worse, in any given month.

In [ ]:
erk.var_historic(hfi, level=5)

### Parametric VaR

The parametric method looks at the price movements of investments over a look-back period and uses probability theory to compute a portfolio's maximum loss. The variance-covariance method for the value at risk calculates the standard deviation of price movements of an investment or security. Assuming stock price returns and volatility follow a normal distribution, the maximum loss within the specified confidence level is calculated.

In [ ]:
erk.var_gaussian(hfi, level=5)

### Cornish-Fisher

The Cornish–Fisher expansion is an asymptotic expansion used to approximate the quantiles of a probability distribution based on its cumulants.

In [ ]:
var_list = [erk.var_gaussian(hfi), erk.var_gaussian(hfi, modified=True), erk.var_historic(hfi)]
comparison = pd.concat(var_list, axis=1)
comparison.columns = ['Gaussian', 'Cornish-Fisher', 'Historic']
comparison.plot.bar(title='Hedge Fund Indices: VaR', figsize=(12,6));

### Conditional VaR

Safer investments like large-cap U.S. stocks or investment-grade bonds rarely exceed VaR by a significant amount. More volatile asset classes, like small-cap U.S. stocks, emerging markets stocks or derivatives, can exhibit CVaRs many times greater than VaRs. Ideally, investors are looking for small CVaRs. However, investments with the most upside potential often have large CVaRs.

In [ ]:
erk.cvar_historic(hfi)


---

# Chapter 2: Efficient Frontier & Markowitz Portfolio

---


# Chapter 2: Efficient Frontier and Markowitz Portfolio

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats 
from pandas_datareader import data 
from datetime import datetime
from scipy.optimize import minimize
import edhec_risk_kit as erk

%load_ext autoreload
%autoreload 2
%matplotlib inline

# using seaborn style (type plt.style.available to see available styles)
plt.style.use("seaborn-pastel")  

## Modern portfolio theory (MPT)

The **Modern portfolio theory (MPT)** is a mathematical framework **for assembling a portfolio of assets such that the expected return is maximized for a given level of volatility**. It is a formalization of **diversification in investing**, i.e., the idea that owning different kinds of financial assets is less risky than owning only one assets. 

### Efficient Frontiers 

In the MPT, the **efficient frontier** is the set of portfolios that can be constructed with the given input assets that have the maximum expected returns for a fixed level of volatility. 

![title](data/efficient_frontier.png)

Hence, suppose that **we have $N > 1$ assets**, and we decide to invest all of our capital in them. Let $\mathbf{w}:=(w_1,\dots,w_N)^T$, with $w_i\in (0,1)$ for all $i=1,\dots,N$, 
be the percentages or the weights of the investments in our portfolio.

Suppose we invest all of our capital, there holds $\sum_{i=1}^N w_i = 1$, and $w_i \ge 0$ (this is a **long-only** strategy).

Let $R_i$ and $R_p$ be the return of asset $i$ and the total return of the portfolio, respectively. 
Likewise, let $\sigma_i$ and $\sigma_p$ be the volatility of asset $i$ and the volatility of the portfolio, respectively. 


### Return of a portfolio
The **total return of the porfolio** is the weigthed sum of the returns of single assets, i.e.,
$$
R_p = \sum_{i=1}^N w_i R_i = \mathbf{w}^T \mathbf{R},
$$
where $\mathbf{R} := (R_1,\dots,R_N)^T$. 



### Volatility of a portfolio

The volatility of the portfolio is then given by:
$$
\sigma_p = \sqrt{ \mathbf{w}^T \Sigma \mathbf{w} }.
$$


### Efficient frontiers of 2-assets portfolios 
In the following, we propose an artificial example in order to see the curve that is drawn by different portfolios which 
are constructed with only $2$ assets with different correlation $\rho_{12}$. 
First of all, we assume to generate $500$ **daily returns** of $2$ assets:

In [ ]:
nret             = 500
periods_per_year = 252
risk_free_rate   = 0.0

Then, we set up a value for the means and the volatility of our two artifical assets:

In [ ]:
mean_1 = 0.001019
mean_2 = 0.001249
vol_1  = 0.016317
vol_2  = 0.019129

and we set up $6$ correlations of the two assets, and for every fixed correlation, 
we will generate $20$ portfolios by allocating $20$ pairs of weights:

In [ ]:
# Correlation goes from 1 (completely correlated) to -1 (conversely correlated)
rhos  = np.linspace(1,-1,num=6) 
ncorr = len(rhos)

# Pairs of weights to be used to construct the portfolios for any given correlation
nweig = 20
w1 = np.linspace(0,1,num=nweig)
w2 = 1 - np.linspace(0,1,num=nweig)
ww = pd.DataFrame( [w1, w2] ).T  

In [ ]:
# Set seed
np.random.seed(1)

# Open the figure
fig, ax = plt.subplots(2,3, figsize=(20,12))    
ax = ax.flatten()

for k_rho, rho in enumerate(rhos):
    # Allocate an empty portfolio 
    portfolio = pd.DataFrame(columns=["return","volatility","sharpe ratio"])

    # Generate the assets' returns with the given correlation rho
    cov_ij     = rho * vol_1 * vol_2
    cov_rets   = pd.DataFrame( [[vol_1**2, cov_ij], [cov_ij, vol_2**2]] )
    daily_rets = pd.DataFrame( np.random.multivariate_normal((mean_1,mean_2), cov_rets.values, nret) )
    
    for i in range(ww.shape[0]):
        # Now, construct the portfolio of two asset with correlation rho and weights ww.loc[i]
        weights = ww.loc[i] 
        # here, weights is a column vector (pd.Series)

        # annualized portfolio returns
        ann_rets      = erk.annualize_rets(daily_rets, periods_per_year)
        portfolio_ret = erk.portfolio_return(weights, ann_rets)        
                
        # annualized portfolio volatility
        portfolio_vol = erk.annualize_vol(np.matmul(daily_rets, weights), periods_per_year)

        # annualized portfolio sharpe ratio
        portfolio_spr = erk.sharpe_ratio(np.matmul(daily_rets, weights), risk_free_rate, periods_per_year)

        # dataframe containing the return, volatility, and the sharpe ratio of the portfolio constructed   
        portfolio = portfolio.append( {"return":portfolio_ret, "volatility": portfolio_vol, "sharpe ratio":portfolio_spr}, ignore_index=True)

    # plot create scatter plot coloured by Sharpe Ratio
    im = ax[k_rho].scatter(portfolio["volatility"]*100, portfolio["return"]*100, c=w2, cmap='RdYlBu') 
    ax[k_rho].grid()
    ax[k_rho].set_title("Correlation: {}".format(np.round(rho,2)), y=0.9, loc='left')
    ax[k_rho].set_xlabel("volatility (%)")
    if k_rho==0: ax[k_rho].set_ylabel("return (%)") 
    ax[k_rho].set_xlim([0,32])
    ax[k_rho].set_ylim([0,95])
    
fig.colorbar(im, ax=ax.ravel().tolist())
plt.show()

For any given correlation, each point represents the pair (return, volatility) of a portfolio constructed with some percentage allocation. 
This can be seen in the colorbar: **red corresponds to $\mathbf{w} = (1,0)$**, i.e., allocation of money only to the first asset, whereas 
**blue corresponds to $\mathbf{w} = (0,1)$**, i.e., allocation of money only ot the second asset.  

We can see that **the lower the correlation between the assets, the better the trade-off between return and volatility**: in this example, 
when $\rho=-1$, we could in principle construct a portfolio which guarantee about $30\%$ of return with almost no volatility.

### Example:  French 30 Industry Portfolios 

In [ ]:
# Monthly returns of thirty different industry portfolios
ind30 = erk.get_ind_returns()
ind30.head()

In [ ]:
# Get expected returns
er = erk.annualize_rets(ind30['1996':'2000'], 12)
er.sort_values().plot.bar(figsize=(12,6))
plt.xlabel('Portfolios')
plt.ylabel('Expected Returns')
plt.title('Industry Expected Returns from 1996 to 2000');

**Calculate the return and volatility given a set of weights and returns.**

In [ ]:
cov = ind30['1996': '2000'].cov()
weights = np.repeat(1/4, 4) # Vector of weights
l = ['Food', 'Beer', 'Smoke', 'Coal'] # Names of the returns
print('Return:', erk.portfolio_return(weights, er[l]))
print('Volatility:', erk.portfolio_vol(weights, cov.loc[l, l]))

### N Asset Frontier

The objective is to maximize the expected return, and minimize volatility. The function `minimize_vol` in the edhec_risk_kit module takes a set of target returns, the returns from a portfolio, and the covariance matrix for that portfolio and calculates a set of weights for various asset allocations such that the volatility for a particular combination is minimized and the return is maximized. Subsequently, the efficient frontier is plotted using `plot_ef` in the edhec_risk_kit module.

In [ ]:
l = ['Smoke', 'Fin', 'Games', 'Coal']
erk.plot_ef(25, er[l], cov.loc[l, l])

## Markowitz Portfolio: CML, MSR, GMV

Recall that a **risk-free asset** is an (hypothetical) asset with a risk-free rate. For example, **short-term government securities (such as US treasury bills)** are used as a risk-free asset since **they pay a fixed interest rate and have exceptionally low default risk**. 

The risk-free asset has zero volatility. Furthermore, it is also uncorrelated with any other asset since, 
by definition, its volatility is zero. Therefore, when combined with any other asset in a portfolio, 
**the change in return is linearly related to the change in risk** as the proportions in the combination vary.

#### The capital market line (CML)

When a risk-free asset is introduced, there will be a line satisfying:

 1. it is tangent to the curve at the risky portfolio with the highest Sharpe ratio; 
 2. its vertical intercept represents a portfolio with $100\%$ of holdings in the risk-free asset; 
 3. the tangency with the curve represents the highest sharpe ratio portfolio with no risk-free holdings and $100%$ of risky assets; 
 assets held in the portfolio occurring at the tangency point; 
 4. points on this line represent portfolios containing positive amounts of both the risky assets and the risk-free asset; 
 
This efficient line is called the **Capital Market Line (CML)**, and its given by:
$$
R_{CML} = R_{f} + \sigma_{CML}\frac{R_{p} - R_{f}}{\sigma_{p}}, 
$$
where $R_p$ and $\sigma_p$ are the return and the volatility of the risky portfolio with no risk free asset, respectively, 
$R_f$ denotes the risk-free rate, and $R_{CML}$ and $\sigma_{CML}$ denote the return and the volatility of the 
portfolio combining both risky assets and the risk-free asset, respectively.

The capital market line (CML) represents portfolios that optimally combine risk and return. Capital asset pricing model (CAPM) depicts the trade-off between risk and return for efficient portfolios. It is a theoretical concept that represents all the portfolios that optimally combine the risk-free rate of return and the market portfolio of risky assets. Under CAPM, all investors will choose a position on the capital market line, in equilibrium, by borrowing or lending at the risk-free rate, since this maximizes return for a given level of risk. The point where the CML is tangent to the efficient frontier is called the Maximum Sharpe Ratio Portfolio.

![title](data/sharpe_portfolio.gif)

Note that the efficient frontier changes shape dramatically when a risk-free asset is introduced.

### Locating the Max Sharpe Ratio (MSR) Portfolio

In [ ]:
ax = erk.plot_ef(20, er, cov)
plt.gca().set_xlim(left=0)
rf = 0.1
w_msr = erk.msr(rf, er, cov)
r_msr = erk.portfolio_return(w_msr, er)
vol_msr = erk.portfolio_vol(w_msr, cov)
cml_x = [0, vol_msr]
cml_y = [rf, r_msr]
plt.gca().plot(cml_x, cml_y, marker='o');

Along the minimum-variance frontier, the left-most point is a portfolio with minimum variance when compared to all possible portfolios of risky assets (shown in red in the plot below). This is known as the global minimum-variance portfolio. An investor cannot hold a portfolio of risky (note: risk-free assets are excluded at this point) assets with a lower risk than the **global minimum-variance (GMV)** portfolio.  

The portion of the minimum-variance curve that lies above and to the right of the global minimum variance portfolio is known as the Markowitz efficient frontier as it contains all portfolios that rational, risk-averse investors would choose.

The function `plot_ef` in the `erk` module allows one to plot the GMV portfolio by setting the `show_gmv` parameter to True and the equal weight portfolio (EW, shown in yellow) by setting `show_ew` to True as well.

In [ ]:
erk.plot_ef(20, er, cov, show_cml=True, riskfree_rate=0.1, show_ew=True, show_gmv=True);

### Danger of MSR Portfolio

Note that very small differences in the expected returns, which are likely present if we attempt to forecast them, can have a dramatic effect on the asset allocation in a portfolio.

In [ ]:
# Calculate the asset allocation for returns in-sample (!)
l = ['Food', 'Steel']
print('Returns:', er[l].round(4).tolist())
print('Portfolio:', erk.msr(0.1, er[l], cov.loc[l, l]).round(4).tolist())

In [ ]:
# Attempt to guess the returns
print('Portfolio:', erk.msr(0.1, np.array([0.11, 0.12]), cov.loc[l, l]).round(4).tolist())
print('Portfolio:', erk.msr(0.1, np.array([0.10, 0.13]), cov.loc[l, l]).round(4).tolist())
print('Portfolio:', erk.msr(0.1, np.array([0.13, 0.10]), cov.loc[l, l]).round(4).tolist())

### Example from the real world: US stocks
Now we will get the timeseries of some US economy stocks and see how to construct a portfolio in an efficient way.

In [ ]:
tickers  = ['AMZN','KO','MSFT']
n_assets = len(tickers) 

stocks = pd.DataFrame()
for stock_name in tickers:
    # daily data
    stocks[stock_name] = data.DataReader(stock_name, data_source="yahoo", 
                                         start=datetime(2010,1,1), end=datetime(2019,11,15))["Adj Close"]    

In [ ]:
stocks.tail()

In [ ]:
# compute the daily returns 
daily_rets = stocks.pct_change().dropna()
daily_rets.tail()

In [ ]:
# compute the mean daily returns and the covariance of daily returns of the two assets
mean_rets = daily_rets.mean()
std_rets  = daily_rets.std()
cov_rets  = daily_rets.cov()
cov_rets

Now we simulate $4000$ portfolios with weights allocated to the stocks above:

In [ ]:
periods_per_year = 252
num_portfolios   = 4000
portfolios       = pd.DataFrame(columns=["return","volatility","sharpe ratio","w1","w2","w3"])
risk_free_rate   = 0

In [ ]:
for i in range(num_portfolios):
    # select random weights
    weights = np.random.random(n_assets)
    # and rescale them to sum to 1
    weights /= np.sum(weights)
    
    # annualized portfolio returns
    ann_rets      = erk.annualize_rets(daily_rets, periods_per_year)
    portfolio_ret = erk.portfolio_return(weights, ann_rets)        

    # annualized portfolio volatility
    portfolio_vol = erk.annualize_vol(daily_rets @ weights, periods_per_year)

    # annualized portfolio sharpe ratio
    portfolio_spr = erk.sharpe_ratio(daily_rets @ weights, risk_free_rate, periods_per_year)
    
    # create dataframe   
    portfolios = portfolios.append( {"return":portfolio_ret, 
                                     "volatility":portfolio_vol, 
                                     "sharpe ratio":portfolio_spr, 
                                     "w1": weights[0], "w2": weights[1], "w3": weights[2]}, 
                                     ignore_index=True)

Now we create a scatter plot coloured by sharpe ratios of the portfolios generated above and we also plot the efficient frontier:

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(10,6)) 

im = ax.scatter(portfolios["volatility"], portfolios["return"], c=portfolios["sharpe ratio"], s=20, edgecolor=None, cmap='RdYlBu')
ax.set_title("Portfolios and efficient frontier")
ax.set_xlabel("volatility")
ax.set_ylabel("return")
ax.grid()
plt.show()

**We will come back to the plot of the efficient frontier below**.

We can see that if the investor is targeting **a return of $20\%$** he could simply hold 
a portfolio **with volatility of about $17.5\%$**, although there are many other portfolios that 
can guarantee the same return but with much higher volatilities such as $22.5\%$. 
It is clear that one should carefully choose the weights of the portfolio. 

We can see, in particular, that there are **two important** portfolios:

1. the **portfolio with the Global Minimum Volatility (GMV)**, i.e., the global minimum variance portfolio
2. the **portfolio with the Maximum Sharpe Ratio (MSR)**.

From the code above we can easily locate these two portfolios in our dataframe by looking at the lowest volatility and highest sharpe ratio 
and and recover the corresponding weights that have been stored.

In [ ]:
# find the portfolio with lowest volatility 
low_vol_portfolio = portfolios.iloc[ portfolios['volatility'].idxmin() ]
print("Global Minimum Volatility portfolio:")
print("- return      : {:.2f}%".format(low_vol_portfolio[0]*100) )
print("- volatility  : {:.2f}%".format(low_vol_portfolio[1]*100) )
print("- sharpe ratio: {:.2f}".format(low_vol_portfolio[2]) )

# find the portfolio with highest sharpe ratio
high_sharpe_portfolio = portfolios.iloc[ portfolios['sharpe ratio'].idxmax() ]
print("\Maximum Sharpe Ratio portfolio:")
print("- return      : {:.2f}%".format(high_sharpe_portfolio[0]*100) )
print("- volatility  : {:.2f}%".format(high_sharpe_portfolio[1]*100) )
print("- sharpe ratio: {:.2f}".format(high_sharpe_portfolio[2]) )

### Finding the optimal portfolios: Global Minimum Volatility Portfolio

In the experiments above, we found the optimal portfolios, i.e., the ones on the efficient frontier, by **simulating a high number of portfolios** and then plotting them. From the plot we could see what the efficient frontier looked like. 
However, we can find an optimal portfolio on the efficient frontier by **solving a minimization problem**, 
for example, by applying the **scipy optimize** method. 


For example, suppose we want to **find the portfolio (on the efficient frontier) which has the minimum volatility**. 
Then the minimization problem is:
$$
\text{minimize} \;\; \frac{1}{2} \mathbf{w}^T\Sigma\mathbf{w}, 
$$
subject to 
$$
\begin{cases}
\mathbf{w}^T \mathbf{1} = 1, \\
0 \leq \mathbf{w} \leq 1.
\end{cases}
$$

In [ ]:
optimal_weights = erk.gmv(cov_rets)
print("optimal weights:")
print("  AMZN: {:.2f}%".format(optimal_weights[0]*100))
print("  KO:   {:.2f}%".format(optimal_weights[1]*100))
print("  MSFT: {:.2f}%".format(optimal_weights[2]*100))

### Finding the optimal portfolios: maximizing the sharpe ratio 

Now, consider the case of finding **the portfolio (on the efficient frontier) with the highest sharpe ratio**. 

Note that scipy offers a **minimize** method, but no a **maximize** a method, and we may then conclude 
that we are not able to find such a portfolio by solving an optimization problem. 
However, **the maximization of the sharpe ratio is nothing but the minimization of the negative sharpe ratio**. 
That is, we have 
$$
\text{minimize} \;\; -  \frac{R_p - r_f}{\sigma_p} =: -\text{SR} 
$$
subject to 
$$
\begin{cases}
\mathbf{w}^T \mathbf{1} = 1, \\
0 \leq \mathbf{w} \leq 1.
\end{cases}
$$

Let us use our minimizer:

In [ ]:
optimal_weights = erk.msr(risk_free_rate, ann_rets, cov_rets)
print("optimal weights:")
print("  AMZN: {:.2f}%".format(optimal_weights[0]*100))
print("  KO:   {:.2f}%".format(optimal_weights[1]*100))
print("  MSFT: {:.2f}%".format(optimal_weights[2]*100))

### Observation on constraints

It is worth mentioning that so far we have decided to invest all of our capital and, at the same time, our strategy has been **long-only**. That is, the weights that we allocate to the assets sum to $1$ 
and all of them are positive (because we **buy** the assets). 
In particular, these two conditions were imposed when solving the minimization problems. 

However, **we could in principle not invest all of our capital**, which means that we at do not necessarily get weights that sum to $1$, and also we may decide to not buy all the assets. We could also **short selling** some of them (by short shelling we mean selling an asset that we do not have and that we borrow from someone else).

#### Short selling and not normalized weigths: minimum volatility portofolio given a fixed return 

We can solve the minimization problem without imposing the constraint on positive weigths and the constraint that the weights sum to $1$, i.e., simply:
$$
\text{minimize} \;\; \frac{1}{2} \mathbf{w}^T\Sigma\mathbf{w}, 
$$
subject to 
$$
\begin{cases}
\mathbf{w}^T \mathbf{R} = R_0, 
\end{cases}
$$
in the case of finding the minimum volatility portfolio for a fixed return. 
In this case we are allowed to **short sell** the asset and in principle we do not have to invest all of our capital.

For such a problem we can find the analytical solution to the problem by using the **Lagrange multipliers**. 
We define the **Lagrangian** of the problem:
$$
\mathcal{L}(\mathbf{w}, \lambda) := \frac{1}{2} \mathbf{w}^T\Sigma\mathbf{w} - \lambda(  \mathbf{w}^T \mathbf{R} - R_0 ),
$$
and put the partial derivatives to zero:
$$
\begin{cases}
\frac{\partial\mathcal{L}}{\partial \mathbf{w}} &= \frac{1}{2} (2\Sigma \mathbf{w}) - \lambda  \mathbf{R} = 0, \\
\frac{\partial\mathcal{L}}{\partial \lambda} &=  - \mathbf{w}^T \mathbf{R} + R_0 = 0.
\end{cases}
$$
From the first equation, we get:
$$
\Sigma \mathbf{w} - \lambda  \mathbf{R} = 0 
\quad\Longrightarrow\quad 
\mathbf{w} = \lambda \Sigma^{-1}\mathbf{R},  
$$
and inserting in the second equation:
$$
- ( \lambda \Sigma^{-1}\mathbf{R} )^T \mathbf{R} + R_0 = 0 
\quad\Longrightarrow\quad 
\lambda \mathbf{R}^T \Sigma^{-1} \mathbf{R} = R_0
\quad\Longrightarrow\quad 
\lambda = \frac{R_0}{\mathbf{R}^T \Sigma^{-1} \mathbf{R}}.
$$
Note that since $\Sigma$ was symmetric, so is $\Sigma^{-1}$, from which $(\Sigma^{-1})^T = \Sigma^{-1}$. 
We can then insert $\lambda$ back into the first equation and obtain:
$$
\mathbf{w}^* = R_0 \frac{\Sigma^{-1}\mathbf{R}}{\mathbf{R}^T \Sigma^{-1} \mathbf{R}},
$$
which is therefore the analytical expression for the weights. Notice that since we have not imposed the constraint 
on the normalisation, we are not guaranteed that such vector of weights sum to $1$. 

#### Short selling and normalized weigths: minimum volatility portofolio given a fixed return 

Analogously, we can also also fin the analytical expression of optimal weights in case we add the condition that the weigths sum to $1$, but without requiring that they have to be all positive, i.e.:
$$
\text{minimize} \;\; \frac{1}{2} \mathbf{w}^T\Sigma\mathbf{w}, 
$$
subject to 
$$
\begin{cases}
\mathbf{w}^T \mathbf{R} &= R_0,  \\
\mathbf{w}^T \mathbf{1} &= 1.
\end{cases}
$$
This is again the case in which we can **short sell** the asset but this time we invest all of the capital. 

We define the Lagrangian:
$$
\mathcal{L}(\mathbf{w}, \lambda) := \frac{1}{2} \mathbf{w}^T\Sigma\mathbf{w} 
- \lambda( \mathbf{w}^T \mathbf{R} - R_0) - \delta(\mathbf{w}^T\mathbf{1}-1),
$$
and put the partial derivatives to zero:
$$
\begin{cases}
\frac{\partial\mathcal{L}}{\partial \mathbf{w}} &= \frac{1}{2} (2\Sigma \mathbf{w}) - \lambda \mathbf{R} - \delta \mathbf{1}= 0, \\
\frac{\partial\mathcal{L}}{\partial \lambda} &=  - \mathbf{w}^T \mathbf{R} + R_0 = 0, \\
\frac{\partial\mathcal{L}}{\partial \lambda} &=  - \mathbf{w}^T \mathbf{1} + R_0 = 0.
\end{cases}
$$
From the first equation we get:
$$
\mathbf{w} = \Sigma^{-1}(\lambda \mathbf{R} + \delta\mathbf{1}), 
$$
and we can insert it in the second and the third equation, respectively:
\begin{cases}
\left(\Sigma^{-1}(\lambda \mathbf{R} + \delta\mathbf{1}) \right)^T\mathbf{R} 
&= \lambda \mathbf{R}^T\Sigma^{-1}\mathbf{R} + \delta\mathbf{1}\Sigma^{-1}\mathbf{R} = R_0, \\
\left(\Sigma^{-1}(\lambda \mathbf{R} + \delta\mathbf{1}) \right)^T\mathbf{1} 
&= \lambda \mathbf{R}^T\Sigma^{-1}\mathbf{1} + \delta\mathbf{1}\Sigma^{-1}\mathbf{1} = 1.
\end{cases}
Let us define the following fixed numbers:
$$
\begin{cases}
A & := \mathbf{R}^T \Sigma^{-1} \mathbf{R},  \\
B & := \mathbf{1}^T \Sigma^{-1} \mathbf{R} \equiv \mathbf{R}^T \Sigma^{-1} \mathbf{1}, \\
C & := \mathbf{1}^T \Sigma^{-1} \mathbf{1},
\end{cases}
$$
where notice that in B the second equation is true since $\Sigma^{-1}$ is a symmetric matrix. Hence we have the following system to solve:
$$
\begin{cases}
\lambda A + \delta B &= R_0, \\
\lambda B + \delta C &= 1.
\end{cases}
$$
From the second equation we find $\lambda$ and put it into the first equation:
$$
\lambda = \frac{1-\delta C}{B}
\quad\Longrightarrow\quad 
\frac{1-\delta C}{B} A + \delta B = R_0
\quad\text{from which we find}\quad
\delta = \frac{R_0B - A}{B^2-AC}.
$$
Now, we put $\delta$ back into $\lambda$:
$$
\lambda = \frac{1 - \frac{R_0 B-A}{B^2-AC}C }{B} = \frac{B - R_0 C}{B^2-AC}.
$$
Finally, we can put both $\lambda$ and $\delta$ we have just find back into $\mathbf{w}$ and find the optimal weight:
$$
\mathbf{w}^*  
= \lambda \Sigma^{-1} \mathbf{R} + \delta \Sigma^{-1} \mathbf{1} 
= \frac{B - R_0 C}{B^2-AC} \Sigma^{-1} \mathbf{R}  +  \frac{R_0B - A}{B^2-AC}  \Sigma^{-1}\mathbf{1} 
= \underbrace{ \frac{1}{B^2-AC}\left(B\Sigma^{-1}\mathbf{R} - A\Sigma^{-1}\mathbf{1} \right) }_{:= \mathbf{f} }
+ R_0 \Bigl( \underbrace{ \frac{1}{B^2-AC}\left(B\Sigma^{-1}\mathbf{1} - C\Sigma^{-1}\mathbf{R} \right) }_{:= \mathbf{g} }  \Bigr)
= \mathbf{f} + R_0 \mathbf{g}.
$$


---

# Chapter 2 (Lab): Portfolio Optimization

---


## Portfolio Optimization

In [ ]:
import pandas as pd
import numpy as np
import edhc_risk_kit as erk
import matplotlib.pyplot as plt
%matplotlib inline
%load_ext autoreload
%autoreload 2

The data used in this notebook is available at http://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html#Research.

In [ ]:
# Monthly returns of thirty different industry portfolios
ind30 = erk.get_ind_returns()
ind30.head()

## Introduction to Optimization and The Efficient Frontier

### Sharpe Ratio

The Sharpe ratio was developed by Nobel laureate William F. Sharpe and is used to help investors understand the return of an investment compared to its risk. The ratio is the average return earned in excess of the risk-free rate per unit of volatility or total risk. Subtracting the risk-free rate from the mean return allows an investor to better isolate the profits associated with risk-taking activities. Generally, the greater the value of the Sharpe ratio, the more attractive the risk-adjusted return.

In [ ]:
# Calculate Sharpe ration with a risk free rate of 3% and 12 periods per year
erk.sharpe_ratio(ind30['1996':'2000'], 0.03, 12).sort_values().plot.bar(figsize=(12,6))
plt.xlabel('Portfolios')
plt.ylabel('Sharpe Ratio')
plt.title('Industry Sharpe Ratios from 1996 to 2000');

### The Efficient Frontier

The efficient frontier is the set of optimal portfolios that offer the highest expected return for a defined level of risk or the lowest risk for a given level of expected return. Portfolios that lie below the efficient frontier are sub-optimal because they do not provide enough return for the level of risk. Portfolios that cluster to the right of the efficient frontier are sub-optimal because they have a higher level of risk for the defined rate of return.  

![title](data/efficient_frontier.png)

In [ ]:
# Get expected returns
er = erk.annualize_rets(ind30['1996':'2000'], 12)
er.sort_values().plot.bar(figsize=(12,6))
plt.xlabel('Portfolios')
plt.ylabel('Expected Returns')
plt.title('Industry Expected Returns from 1996 to 2000');

In order to calculate the efficient frontier, we need to have an estimate of the expected returns and the covariance matrix for the set of risky securities which will used to build the optimal portfolio.  These parameters are difficult (impossible) to forecast, and the optimal portfolio calculation is extremely sensitive to these parameters.  

The function `portfolio_return` returns the matrix muliplication between the transposed vector of weights and the returns. The function `portfolio_vol` calculates the matrix multiplication between the transposed vector of weights, the covariance matrix of the returns, and the weights.

In [ ]:
cov = ind30['1996': '2000'].cov()

**Calculate the return and volatility given a set of weights and returns.**

In [ ]:
weights = np.repeat(1/4, 4) # Vector of weights
l = ['Food', 'Beer', 'Smoke', 'Coal'] # Names of the returns
print('Return:', erk.portfolio_return(weights, er[l]))
print('Volatility:', erk.portfolio_vol(weights, cov.loc[l, l]))

### Two Asset Frontier

In [ ]:
l = ['Fin', 'Beer']
erk.plot_ef2(25, er[l], cov.loc[l, l]);

### N Asset Frontier

The objective is to maximize the expected return, and minimize volatility. The function `minimize_vol` in the edhec_risk_kit module takes a set of target returns, the returns from a portfolio, and the covariance matrix for that portfolio and calculates a set of weights for various asset allocations such that the volatility for a particular combination is minimized and the return is maximized. Subsequently, the efficient frontier is plotted using `plot_ef` in the edhec_risk_kit module.

In [ ]:
l = ['Smoke', 'Fin', 'Games', 'Coal']
erk.plot_ef(25, er[l], cov.loc[l, l])

## Implementing Markowitz

The capital market line (CML) represents portfolios that optimally combine risk and return. Capital asset pricing model (CAPM) depicts the trade-off between risk and return for efficient portfolios. It is a theoretical concept that represents all the portfolios that optimally combine the risk-free rate of return and the market portfolio of risky assets. Under CAPM, all investors will choose a position on the capital market line, in equilibrium, by borrowing or lending at the risk-free rate, since this maximizes return for a given level of risk. The point where the CML is tangent to the efficient frontier is called the Maximum Sharpe Ratio Portfolio.

![title](data/sharpe_portfolio.gif)

Note that the efficient frontier changes shape dramatically when a risk-free asset is introduced.

### Locating the Max Sharpe Ratio Portfolio

In [ ]:
ax = erk.plot_ef(20, er, cov)
plt.gca().set_xlim(left=0)
rf = 0.1
w_msr = erk.msr(rf, er, cov)
r_msr = erk.portfolio_return(w_msr, er)
vol_msr = erk.portfolio_vol(w_msr, cov)
cml_x = [0, vol_msr]
cml_y = [rf, r_msr]
plt.gca().plot(cml_x, cml_y, marker='o');

Along the minimum-variance frontier, the left-most point is a portfolio with minimum variance when compared to all possible portfolios of risky assets (shown in red in the plot below). This is known as the global minimum-variance portfolio. An investor cannot hold a portfolio of risky (note: risk-free assets are excluded at this point) assets with a lower risk than the global minimum-variance (GMV) portfolio.  

The portion of the minimum-variance curve that lies above and to the right of the global minimum variance portfolio is known as the Markowitz efficient frontier as it contains all portfolios that rational, risk-averse investors would choose.

The function `plot_ef` in the `erk` module allows one to plot the GMV portfolio by setting the `show_gmv` parameter to True and the equal weight portfolio (EW, shown in yellow) by setting `show_ew` to True as well.

In [ ]:
erk.plot_ef(20, er, cov, show_cml=True, riskfree_rate=0.1, show_ew=True, show_gmv=True);

Note that very small differences in the expected returns, which are likely present if we attempt to forecast them, can have a dramatic effect on the asset allocation in a portfolio.

In [ ]:
# Calculate the asset allocation for returns in-sample (!)
l = ['Food', 'Steel']
print('Returns:', er[l].round(4).tolist())
print('Portfolio:', erk.msr(0.1, er[l], cov.loc[l, l]).round(4).tolist())

In [ ]:
# Attempt to guess the returns
print('Portfolio:', erk.msr(0.1, np.array([0.11, 0.12]), cov.loc[l, l]).round(4).tolist())
print('Portfolio:', erk.msr(0.1, np.array([0.10, 0.13]), cov.loc[l, l]).round(4).tolist())
print('Portfolio:', erk.msr(0.1, np.array([0.13, 0.10]), cov.loc[l, l]).round(4).tolist())

### Exercises

Use the EDHEC Hedge Fund Indices data set. Perform the following analysis based on data since 2000 (including all of 2000)

**What was the Monthly Parametric Gaussian VaR at the 1% level (as a +ve number) of the Distressed Securities strategy?**

In [ ]:
hfi = erk.get_hfi_returns()
(erk.var_gaussian(hfi['2000':]['Distressed Securities'], level=1)*100).round(2)

**What was the 1% VaR for the same strategy after applying the Cornish-Fisher Adjustment?**

In [ ]:
(erk.var_gaussian(hfi['2000':]['Distressed Securities'], level=1, modified=True)*100).round(2)

**What was the Monthly Historic VaR at the 1% level (as a +ve number) of the Distressed Securities strategy?**

In [ ]:
(erk.var_historic(hfi['2000':]['Distressed Securities'], level=1)*100).round(2)

Using data during the 5 year period 2013-2017 (both inclusive) for the 30 industry return data, estimate the expected returns as well as the covariance matrix for the `Books`, `Steel`, `Oil`, and `Mines` industries. Assume the risk free rate over the 5 year period is 10%.

In [ ]:
l = ['Books', 'Steel', 'Oil', 'Mines']

In [ ]:
cov = ind30['2013':'2017'].cov()

**What is the weight of Steel in the EW Portfolio?**

In [ ]:
print(0.25)

**What is the weight of the largest component of the MSR portfolio?**

In [ ]:
msr_weights = erk.msr(0.1, er[l], cov.loc[l, l])
print(msr_weights.max(), l[msr_weights.argmax()])

**What is the weight of the largest component of the GMV portfolio?**

In [ ]:
gmv_weights = erk.gmv(cov.loc[l, l])
print(gmv_weights.max(), l[gmv_weights.argmax()])


---

# Chapter 3 (Lab): Limits of Diversification

---


# Limits of Diversification

In [ ]:
import pandas as pd
import numpy as np
import edhc_risk_kit as erk
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline
%load_ext autoreload
%autoreload 2

With diversification, an investor attempts to create a portfolio that includes multiple investments in order to reduce risk. Consider, for example, an investment that consists of only stock issued by a single company. If that company's stock suffers a serious downturn, your portfolio will sustain the full brunt of the decline. By splitting your investment between the stocks from two different companies, you can reduce the potential risk to your portfolio.

Hedging can be thought of as insurance. When people decide to hedge, they are insuring themselves against a negative event. This doesn't prevent a negative event from happening, but if it does happen and you're properly hedged, the impact of the event is reduced. 

Portfolio managers, individual investors, and corporations use hedging techniques to reduce their exposure to various risks. In financial markets, however, hedging becomes more complicated than simply paying an insurance company a fee every year. Hedging against investment risk means strategically using instruments in the market to offset the risk of any adverse price movements. In other words, investors hedge one investment by making another. Technically, to hedge you would invest in two securities with negative correlations. 

In [ ]:
ind_return = erk.get_ind_file('returns') # Get monthly industry returns
ind_size = erk.get_ind_file('size')      # Get market size of firms
ind_nfirms = erk.get_ind_file('nfirms')  # Get number of firms
print(ind_return.shape, ind_size.shape, ind_nfirms.shape)

In [ ]:
ind_mktcap = ind_nfirms * ind_size
total_mktcap = ind_mktcap.sum(axis='columns')
total_mktcap.plot(figsize=(12,6));

Clearly, the market has grown over time.

# Implementing a Cap-Weighted Portfolio

### The Capitalization-Weighted Index

A capitalization-weighted index is a type of market index with individual components, or securities, weighted according to their total market capitalization. Market capitalization uses the total market value of a firm's outstanding shares. The calculation multiples outstanding shares by the current price of a single share. The components with a higher market cap carry a higher weighting percentage in the index.  

One way to construct a market index is to build a portfolio whose weights are rebalanced back to the target weights every period. In reality, the weights may drift over time but this simplification is close enough.

In [ ]:
# Comparison of industries
ind_capweight = ind_mktcap.divide(total_mktcap, axis='rows')
ind_capweight[['Fin', 'Steel']].plot(figsize=(12,6));

In [ ]:
# Calculate total market return
total_market_return = (ind_capweight * ind_return).sum(axis='columns')
# Calculate market index
total_market_index = erk.drawdown(total_market_return).Wealth
total_market_index.plot(figsize=(12,6), title='Total Market Capweighted Index 1926-2018');

In [ ]:
# Plot the index and a moving average
total_market_index['1980':].plot(figsize=(12,6))
total_market_index['1980':].rolling(window=36).mean().plot();

Let's create a time series of the annualized returns over the trailing 36 months and the average correlation across stocks over that same 36 months.

In [ ]:
# Compute trailing 36 months returns
tmi_tr36rets = total_market_return.rolling(window=36).aggregate(erk.annualize_rets, periods_per_year=12)
tmi_tr36rets.plot(figsize=(12,6), label='Trailing 36 Month Return', legend=True);
total_market_return.plot(label='Total Market Return', legend=True);

Next we want to look at average correlations between all the industries over that same trailing 3 year window.

In [ ]:
ts_corr = ind_return.rolling(window=36).corr()
ts_corr.index.names = ['date', 'industry']
ind_tr36corr = ts_corr.groupby(level='date').apply(lambda cormat: cormat.values.mean())
tmi_tr36rets.plot(secondary_y=True, legend=True, label="Tr 36 mo return", figsize=(12,6))
ind_tr36corr.plot(legend=True, label="Tr 36 mo Avg Correlation");

In [ ]:
tmi_tr36rets['2007':].plot(secondary_y=True, legend=True, label="Tr 36 mo return", figsize=(12,6))
ind_tr36corr['2007':].plot(legend=True, label="Tr 36 mo Avg Correlation");

In [ ]:
print(tmi_tr36rets.corr(ind_tr36corr)) # Correlation between returns and average correlation
print(tmi_tr36rets['2007':].corr(ind_tr36corr['2007':]))

Clearly, these two series are negatively correlated, which explains why diversification fails you when you need it most. When markets fall, correlations rise, making diversification much less valuable.

# Implementing Portfolio Insurance (CPPI) and Drawdown Constraints

Constant Proportion Portfolio Insurance (CPPI) is a type of portfolio insurance in which the investor sets a floor on the dollar value of their portfolio, then structures asset allocation around that decision. The two asset classes used in CPPI are a risky asset (usually equities or mutual funds) and a conservative asset of either cash, equivalents or treasury bonds. The percentage allocated to each depends on the "cushion" value, defined as current portfolio value minus floor value, and a multiplier coefficient, where a higher number denotes a more aggressive strategy.

In [ ]:
# Load the industry returns and the total market index we previously created
ind_return = erk.get_ind_returns()
tmi_return = erk.get_total_market_index_returns()

The CPPI algorithm is surprisingly simple to implement. It takes as input the returns of a risky asset and a safe asset, the initial wealth to invest at the start, along with a floor that should not be violated.

In [ ]:
risky_r = ind_return["2000":][["Steel", "Fin", "Beer"]]
# Assume the safe asset is paying 3% per year
safe_r = pd.DataFrame().reindex_like(risky_r)
safe_r.values[:] = 0.03/12 # fast way to set all values to a number
start = 1000 # start at $1000
floor = 0.80 # set the floor to 80 percent of the starting value

Now, we need to backtest this strategy by starting at the beginning and at every point in time:

1. Compute the cushion (asset value minus floor)
2. Compute the allocation (based on the multiplier)
3. Compute the new asset value

The function `run_cppi` in the `edhec_risk toolkit` implements the following steps necessary to calculate the CPPI. Note that the floor is not updated at this point.

In [ ]:
# set up the CPPI parameters
dates = risky_r.index
n_steps = len(dates)
account_value = start
floor_value = start*floor
m = 3

## set up some DataFrames for saving intermediate values
account_history = pd.DataFrame().reindex_like(risky_r)
risky_w_history = pd.DataFrame().reindex_like(risky_r)
cushion_history = pd.DataFrame().reindex_like(risky_r)

In [ ]:
for step in range(n_steps):
    cushion = (account_value - floor_value)/account_value
    risky_w = m * cushion
    risky_w = np.minimum(risky_w, 1)
    risky_w = np.maximum(risky_w, 0)
    safe_w = 1 - risky_w
    risky_alloc = account_value * risky_w
    safe_alloc = account_value * safe_w
    # recompute the new account value at the end of this step
    account_value = risky_alloc * (1 + risky_r.iloc[step]) + safe_alloc * (1 + safe_r.iloc[step])
    # save the histories for analysis and plotting
    cushion_history.iloc[step] = cushion
    risky_w_history.iloc[step] = risky_w
    account_history.iloc[step] = account_value
    risky_wealth = start * (1 + risky_r).cumprod()

Compare investment strategy with or without CPPI for different industries.

In [ ]:
ind = "Beer"
ax = account_history[ind].plot(figsize=(10,5))
risky_wealth[ind].plot(style="k:")
ax.axhline(y=floor_value, color='r', linestyle='--');

In [ ]:
ind = "Fin"
ax = account_history[ind].plot(figsize=(10,5))
risky_wealth[ind].plot(style="k:")
ax.axhline(y=floor_value, color='r', linestyle='--');

In [ ]:
# Example
btr = erk.run_cppi(ind_return["2007":][["Steel", "Fin", "Beer"]])
ax = btr["Wealth"].plot(figsize=(12,5))
btr["Risky Wealth"].plot(ax=ax, style="--");

The function `summary_stats` is a convenience function in the `edhic_risk_kit` which provides summary statistics on a set of returns.

In [ ]:
erk.summary_stats(btr["Risky Wealth"].pct_change().dropna())

Insurance strategies usually help with drawdowns, but they can also be adapted to explictly limit the drawdown.

## Explicitly Limiting Drawdowns

In [ ]:
btr = erk.run_cppi(ind_return["2007":][["Steel", "Fin", "Beer"]], drawdown=0.25)
ax = btr["Wealth"].plot(figsize=(12,5))
btr["Risky Wealth"].plot(ax=ax, style="--")

In [ ]:
erk.summary_stats(btr["Wealth"].pct_change().dropna())[["Annualized Return",
                                                        "Annualized Vol",
                                                        "Sharpe Ratio",
                                                        "Max Drawdown"]]

# Random Walk Generation

The stochastic model for asset returns is given by the Geometric Brownian Motion (GBM) process:

$$ \frac{S_{t+dt}-S_t}{S_t} = \mu dt + \sigma\sqrt{dt}\xi_t $$

When we generate simulated returns, we can usually ignore the decomposition of $\mu$ since we only care about the net effective drift term without worrying about what the components of it are. The function `gbm` in the `edhc_risk_kit` implements the GBM process. The function takes as inputs the number of years you want to generate returns and the number of scenarios as well as the parameters for the model, i.e., $\mu$, $\sigma$, the step size (e.g., per year) and the initial stock price.

In [ ]:
# Example
erk.gbm(n_years=5, n_scenarios=5, mu=0.07).plot(figsize=(12,5), legend=False);

In [ ]:
def show_gbm(n_scenarios, mu, sigma):
    """
    Draw the results of a stock price evolution under a Geometric Brownian Motion model
    """
    s_0=100
    prices = erk.gbm(n_scenarios=n_scenarios, mu=mu, sigma=sigma, s_0=s_0)
    ax = prices.plot(legend=False, color="indianred", alpha = 0.5, linewidth=2, figsize=(12,5))
    ax.axhline(y=100, ls=":", color="black")
    # draw a dot at the origin
    ax.plot(0,s_0, marker='o',color='darkred', alpha=0.2)

In [ ]:
gbm_controls = widgets.interactive(show_gbm, 
                                   n_scenarios=widgets.IntSlider(min=1, max=1000, step=1, value=1), 
                                   mu=(0., +.2,.01),
                                   sigma=(0, .3, .01))
display(gbm_controls)

# Interactive Plotting and Monte Carlo Simulations of CPPI

In [ ]:
def show_cppi(n_scenarios=50, mu=0.07, sigma=0.15, m=3, floor=0., riskfree_rate=0.03, y_max=100):
    """
    Plot the results of a Monte Carlo Simulation of CPPI
    """
    start = 100
    sim_rets = erk.gbm(n_scenarios=n_scenarios, mu=mu, sigma=sigma, prices=False, steps_per_year=12)
    risky_r = pd.DataFrame(sim_rets)
    # run the "back"-test
    btr = erk.run_cppi(risky_r=pd.DataFrame(risky_r),riskfree_rate=riskfree_rate,m=m, start=start, floor=floor)
    wealth = btr["Wealth"]
    y_max=wealth.values.max()*y_max/100
    ax = wealth.plot(legend=False, alpha=0.3, color="indianred", figsize=(12, 6))
    ax.axhline(y=start, ls=":", color="black")
    ax.axhline(y=start*floor, ls="--", color="red")
    ax.set_ylim(top=y_max)

cppi_controls = widgets.interactive(show_cppi, 
                                   n_scenarios=widgets.IntSlider(min=1, max=1000, step=5, value=50), 
                                   mu=(0., +.2, .01),
                                   sigma=(0, .30, .05),
                                   floor=(0, 2, .1),
                                   m=(1, 5, .5),
                                   riskfree_rate=(0, .05, .01),
                                   y_max=widgets.IntSlider(min=0, max=100, step=1, value=100,
                                                          description="Zoom Y Axis"))
display(cppi_controls)

# Adding a Histogram and Reporting Floor Violations

In [ ]:
def show_cppi(n_scenarios=50, mu=0.07, sigma=0.15, m=3, floor=0., riskfree_rate=0.03, y_max=100):
    """
    Plot the results of a Monte Carlo Simulation of CPPI
    """
    start = 100
    sim_rets = erk.gbm(n_scenarios=n_scenarios, mu=mu, sigma=sigma, prices=False, steps_per_year=12)
    risky_r = pd.DataFrame(sim_rets)
    # run the "back"-test
    btr = erk.run_cppi(risky_r=pd.DataFrame(risky_r),riskfree_rate=riskfree_rate,m=m, start=start, floor=floor)
    wealth = btr["Wealth"]
    # calculate terminal wealth stats
    y_max=wealth.values.max()*y_max/100
    terminal_wealth = wealth.iloc[-1]
    # Plot!
    fig, (wealth_ax, hist_ax) = plt.subplots(nrows=1, ncols=2, sharey=True, gridspec_kw={'width_ratios':[3,2]}, figsize=(24, 9))
    plt.subplots_adjust(wspace=0.0)
    
    wealth.plot(ax=wealth_ax, legend=False, alpha=0.3, color="indianred")
    wealth_ax.axhline(y=start, ls=":", color="black")
    wealth_ax.axhline(y=start*floor, ls="--", color="red")
    wealth_ax.set_ylim(top=y_max)
    
    terminal_wealth.plot.hist(ax=hist_ax, bins=50, ec='w', fc='indianred', orientation='horizontal')
    hist_ax.axhline(y=start, ls=":", color="black")

cppi_controls = widgets.interactive(show_cppi, 
                                   n_scenarios=widgets.IntSlider(min=1, max=1000, step=5, value=50), 
                                   mu=(0., +.2, .01),
                                   sigma=(0, .3, .05),
                                   floor=(0, 2, .1),
                                   m=(1, 5, .5),
                                   riskfree_rate=(0, .05, .01),
                                   y_max=widgets.IntSlider(min=0, max=100, step=1, value=100,
                                                          description="Zoom Y Axis")
)
display(cppi_controls)

# Adding Terminal Wealth Statistics

In [ ]:
def show_cppi(n_scenarios=50, mu=0.07, sigma=0.15, m=3, floor=0., riskfree_rate=0.03, steps_per_year=12, y_max=100):
    """
    Plot the results of a Monte Carlo Simulation of CPPI
    """
    start = 100
    sim_rets = erk.gbm(n_scenarios=n_scenarios, mu=mu, sigma=sigma, prices=False, steps_per_year=steps_per_year)
    risky_r = pd.DataFrame(sim_rets)
    # run the "back"-test
    btr = erk.run_cppi(risky_r=pd.DataFrame(risky_r),riskfree_rate=riskfree_rate,m=m, start=start, floor=floor)
    wealth = btr["Wealth"]

    # calculate terminal wealth stats
    y_max=wealth.values.max()*y_max/100
    terminal_wealth = wealth.iloc[-1]
    
    tw_mean = terminal_wealth.mean()
    tw_median = terminal_wealth.median()
    failure_mask = np.less(terminal_wealth, start*floor)
    n_failures = failure_mask.sum()
    p_fail = n_failures/n_scenarios

    e_shortfall = np.dot(tv-start*floor, failure_mask)/n_failures if n_failures > 0 else 0.0

    # Plot!
    fig, (wealth_ax, hist_ax) = plt.subplots(nrows=1, ncols=2, sharey=True, gridspec_kw={'width_ratios':[3,2]}, figsize=(24, 9))
    plt.subplots_adjust(wspace=0.0)
    
    wealth.plot(ax=wealth_ax, legend=False, alpha=0.3, color="indianred")
    wealth_ax.axhline(y=start, ls=":", color="black")
    wealth_ax.axhline(y=start*floor, ls="--", color="red")
    wealth_ax.set_ylim(top=y_max)
    
    terminal_wealth.plot.hist(ax=hist_ax, bins=50, ec='w', fc='indianred', orientation='horizontal')
    hist_ax.axhline(y=start, ls=":", color="black")
    hist_ax.axhline(y=tw_mean, ls=":", color="blue")
    hist_ax.axhline(y=tw_median, ls=":", color="purple")
    hist_ax.annotate(f"Mean: ${int(tw_mean)}", xy=(.7, .9),xycoords='axes fraction', fontsize=24)
    hist_ax.annotate(f"Median: ${int(tw_median)}", xy=(.7, .85),xycoords='axes fraction', fontsize=24)
    if (floor > 0.01):
        hist_ax.axhline(y=start*floor, ls="--", color="red", linewidth=3)
        hist_ax.annotate(f"Violations: {n_failures} ({p_fail*100:2.2f}%)\nE(shortfall)=${e_shortfall:2.2f}", xy=(.7, .7), xycoords='axes fraction', fontsize=24)

cppi_controls = widgets.interactive(show_cppi,
                                   n_scenarios=widgets.IntSlider(min=1, max=1000, step=5, value=50), 
                                   mu=(0., +.2, .01),
                                   sigma=(0, .3, .05),
                                   floor=(0, 2, .1),
                                   m=(1, 5, .5),
                                   riskfree_rate=(0, .05, .01),
                                   steps_per_year=widgets.IntSlider(min=1, max=12, step=1, value=12,
                                                          description="Rebals/Year"),
                                   y_max=widgets.IntSlider(min=0, max=100, step=1, value=100,
                                                          description="Zoom Y Axis")
)
display(cppi_controls)


---

# Chapter 3: Constant Proportion Portfolio Insurance (CPPI)

---


# Chapter 3: Constant Proportion Portfolio Insurance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats 
from pandas_datareader import data 
from datetime import datetime
from scipy.optimize import minimize
import edhec_risk_kit as erk

# using seaborn style (type plt.style.available to see available styles)
plt.style.use("seaborn-pastel") 

%load_ext autoreload
%autoreload 2
%matplotlib inline

## The limits of Portfolio diversification


Diversification is an **effective way** to decrease **idiosyncratic or specific** risk.


However, diversification is **not effective** to decrease the **systemic risk**: 
 
We would show that in case of the financial crisis, diversification of 
a portfolio is not a guarantee of less risk.

The dataframe **ind_rets** contains the returns (from 1926-2018) of 30 portfolios including different industry sectors such as food, beer, smoke, etc. 

In each portofolio, each asset is weighted by its market capitalization.

In [ ]:
ind_rets   = erk.get_ind_file(filetype="returns")
ind_nfirms = erk.get_ind_file(filetype="nfirms")
ind_size   = erk.get_ind_file(filetype="size")
ind_rets.head(3)

**The number of firms** composing each single sector are stored in **ind_nfirms**:

In [ ]:
ind_nfirms.head(3)

that is, in 1926-07, there were 43 (food) companies in the Food portfolio, 3 (beer) companies in the Beer portfolio, etc.

Finally, the datframe **ind_size** contains the **average size** of the companies composing the portfolio:

In [ ]:
ind_size.head(3)

that is, the average size of the 43 Food companies in 1926-07 was 35.98, the average size of the 3 Beer companies was 7.12, ans so on (the unit does not matter). 

### Constructing the index

The first thing to do is to get the **market capitalization of each industry sector.**

In [ ]:
ind_mkt_cap = ind_nfirms * ind_size
ind_mkt_cap.head(3)

Now, we want to get the **total market capitalization**, in order to get the fraction of the total market capitalization of each industry. 

In [ ]:
# total market capitalization: 
total_mkt_cap = ind_mkt_cap.sum(axis=1)
total_mkt_cap.head()

and now, we can get the weight of each industry by dividing the market-cap of each industry by the total market cap:

In [ ]:
ind_cap_weights = ind_mkt_cap.divide(total_mkt_cap, axis=0)
ind_cap_weights.head(3)

Let us visualize these things:

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(18,4)) 
total_mkt_cap.plot(grid=True, ax=ax[0]) 
ax[0].set_title("Total market cap 1929-2018")

ind_cap_weights[["Steel","Fin","Telcm"]].plot(grid=True, ax=ax[1])
ax[1].set_title("Steel, Finance, and Telecommunication Market caps (%) 1929-2018")
plt.show()

In the plot on the left we have the total market cap from 1929 to 2018. On the right, we have the Steel, Finance, and Telecommunication Market 
caps 1929-2018 as a percentage of the total market cap. 

For example, notice that while the Finance sector was around the 3% of the total market cap in 1929, in 2018 it was more than 15%. On the other hand, the Steel sector was around the 9% in 1929 until decreasing to the 0.2% in 2018.

Now, let us find the **total maket return**, i.e., the return time series from the total market. 

Once we have the total market return series, we can **compound** it and obtain 
the **total market index** (what we called the **wealth** in week 1):

In [ ]:
total_market_return = (ind_cap_weights * ind_rets).sum(axis=1)

# suppose to have invested a capital of 1000$ in the market
capital = 1000
total_market_index = capital * (1 + total_market_return).cumprod()

fig, ax = plt.subplots(1,2,figsize=(18,4)) 

total_market_index.plot(grid=True, ax=ax[0]) 
ax[0].set_title("Total market cap-weigthed index 1929-2018")

total_market_return.plot(grid=True, ax=ax[1])
ax[1].set_title("Total market cap-weigthed returns 1929-2018")

plt.show()

Next, we what to see **how returns are related to market correlations**.

### Rolling returns

Let us plot the total market index, say from 1990, and plot together some moving average (MA) series corresponding to 60, 36, and 12 months (i.e, 5, 3, and 1 years), respectively, just to look at how the **.rolling** method works.

In [ ]:
total_market_index["1990":].plot(grid=True, figsize=(11,6), label="Total market cap-weighted index")

total_market_index["1990":].rolling(window=60).mean().plot(grid=True, figsize=(11,6), label="60 months MA") # 5 years MA
total_market_index["1990":].rolling(window=36).mean().plot(grid=True, figsize=(11,6), label="36 months MA") # 3 years MA
total_market_index["1990":].rolling(window=12).mean().plot(grid=True, figsize=(11,6), label="12 months MA") # 1 year MA

plt.legend()
plt.show()

Now compute the **trailing 36 months compound returns of total_market_return**. 
That is, we take the total market return series and open the rolling windows for 36 months. Then, for each rolling window (by using **.aggregate**) we compound the returns: 

In [ ]:
tmi_trail_36_rets = total_market_return.rolling(window=36).aggregate( erk.annualize_rets, periods_per_year=12 )

In [ ]:
# Plot them
total_market_return.plot(grid=True, figsize=(12,5), label="Total market (monthly) return")
tmi_trail_36_rets.plot(grid=True, figsize=(12,5), label="Trailing 36 months total market compound return")
plt.legend()
plt.show()

### Rolling correlations: multi-indices and groupby

Let us now compute the **rolling** correlations across industries, in the same way as we have computed the trailing 36 months compound returns. We use the **.corr()** method which computes the **pairwise correlation** between columns of the dataframe.

In [ ]:
rets_trail_36_corr = ind_rets.rolling(window=36).corr()
rets_trail_36_corr.index.names = ["date","industry"]
rets_trail_36_corr.tail()

The dataframe above is a **time series of correlation matrices**. 
That is, each matrix represents the trailing $36$ months **correlation matrix** of compounded 
returns of the industries for each available data. 
That is, if we look at **rets_trail_36_corr.tail(30)** we would see the (trailing 36 months) correlation matrix of industries during the last available month, i.e., 2018-12,. 
If we look at **rets_trail_36_corr.tail(60)** we would see the same correlation matrix for 2018-12 and the (trailing 36 months) correlation matrix of 2018-11, etc.

In the example above, we see that during 2018-12, the  (trailing 36 months) correlation between Whlsl and Food was about 0.47, 
the (trailing 36 months) correlation betwenn Rtail and Smoke was about 0.03, and so on. 

Notice that due to the structure of such a dataframe, we have a **double index**: per each index date, we have the set of index industries. 

Next, we want to the see **averages of all these correlation matrices** for each date. To do that, we first get single correlation matrices  using **groupby by date** and the we take the average of them. The result is a pd.Series:

In [ ]:
ind_trail_36_corr = rets_trail_36_corr.groupby(level="date").apply(lambda corrmat: corrmat.values.mean())

In [ ]:
fig, ax1 = plt.subplots(1,1,figsize=(14,6))

tmi_trail_36_rets.plot(ax=ax1, color="blue", grid=True, label="Trailing 36 months total market compound return")
ax2 = ax1.twinx()
ind_trail_36_corr.plot(ax=ax2, color="orange", grid=True, label="Trailing 36 months total market return correlations")

ax1.set_ylabel('trail 36mo returns')
ax2.set_ylabel('trail 36mo corrs',rotation=-90)
ax1.legend(loc=2)
ax2.legend(loc=1)
plt.show()

For example, during the 1929 crisis, we see that when (trailing 36 months) returns of the overall market falls, the average correlation of industries of the market increases, and that when returns recover, then correlations fall. 
This can be also seen in the Lehman Brother crisis. When in 2007 returns started to fall then correlations increased.

When this happens, we simply realize **the limits of diversification**. Returns from the entire market falls means that, on average, all industries in the market are suffering losses, i.e., their correlation increases, and it does not really matter how diversified the portfolio is. **Diversification may not help you when market crashes**.

However, **this is not always so: look at the dot com crisis of from 1999.** Returns and correlations follow the same trend, showing that in this case diversification could help to stop losses$\dots$

In general:

In [ ]:
tmi_trail_36_rets.corr(ind_trail_36_corr)

this is, the correlation between the series of (trailing 36 months) compounded returns and the series of average correlations 
across industries **is negative**.

## Risk insurance strategies 

Recent market crises have highlighted the benefits of risk-controlled strategies that allow investors to benefit from **downside risk protection**. 
CPPI is one of the risk insurance strategies that aims at achieving the highest possible upside potential while imposing stringent limits to portfolio drawdown.

### Constant Proportion Portfolio Insurance (CPPI)

The **CPPI** procedure allows to obtain option-like (convex) payoffs without using options. 





**Strategy:**
- Two asset classes
    - **risky asset** (e.g. equities or mutual funds)
    - **conservative asset** (e.g. cash or treasury bonds). 
- **Dynamic allocation** between the risky and the safe asset. 
    - **Cushion value:** current portfolio value - **floor value**. 
    - Allocate to a risky asset **a positive integer number $m$ times the cushion**
    - Allocate the remaining proportion to the safe asset. 

When the cushion goes to zero, i.e., our allocation to such asset also goes to zero by definition.

**Example**: suppose we want to invest in a risky asset and we choose $m=4$ and a floor of $90\%$ of our portfolio. 
Then, our allocation should be $4(1-0.9)\% = 40\%$ of our total capital to the risky asset (and then $60\%$ to the safe asset).

#### Allocation to the risky asset

Let us denote as $A$ the value of our portfolio that we will call the **account value** (in general, is the capital invested). 
Hence, the allocation to the risky asset has to be:
$$
E := m (A-F) = m A (1 - f), 
$$
where $F:=fA$ is the **floor value** since $f$ denotes the percentage of our wealth we want to preserve (i.e., $0.9$ in the example above).

It turns out that **the multiplier $m$ should be chosen proportionally to the drop in value we are ready to take**. 
For example, let us suppose that $F$ is our floor value, 
and that we have invested a quantity $E$ out of our account value $A$ to the risky asset. 
Suppose that during time the risky asset drops by $d\%$. **What was the value of $m$ that did not break the floor?**

Basically, we want to see which value of $m$ satisfies:
$$
\text{account}-\text{loss} = A - dE \geq F = \text{floor value},
$$
where $dE$ is the drop in value of our investment in the risky asset (the **loss**) and $F$ is the floor value. 
If we substitute, we obtain:
$$
A - dmA(1-f) \geq fA
$$
and then:
$$
1 - dm(1-f) \geq f 
\qquad\Longrightarrow\qquad
m \leq \frac{1}{d}.
$$
That is, if we choosed $m$ less than or equal to $1/d$ we did not go below the floor. 

Notice that **if $m = 1/d$**, then 
$$
\text{account}-\text{loss} = A - dE = A - d\frac{1}{d}A(1-f) = fA = \text{floor value},
$$
that is, we basically loose the entire drop and reach the floor. By choosing $m=6$, with such a drop, 
we would loose more than the $20\%$ and go below the floor.


**Example**: let the drop be $20\%$. Then
$$
m \leq \frac{1}{0.2} = 5,
$$
that is, we should choose a multiplier at most equal to $5$ in order to not go below the floor.

**In general: if $m\%$ is the maximum drop that we are ready to take then the multiplier should be (at most) $1/m\%$**.

### Implementing CPPI with Drawdown constraint

An algorithm using the CPPI strategy is based on the following three simple things:

1. Computing the **cushion** (Account value minus Floor value)
2. Compute the allocation to both the risky and the safe asset
3. Update the account value based on returns

Let us create this investment strategy using the industry returns and the total market index returns.

In [ ]:
ind_return = erk.get_ind_file(filetype="returns")
tmi_return = erk.get_total_market_index_returns()

Let us consider the industry returns from 2000 and let us get only three industries. 
These are going to be our risky assets:

In [ ]:
risky_rets = ind_return["2000":][["Steel","Fin","Beer"]]

For the safe asset, let us just create an artificial set of assets which guarantee an annual return of $3\%$:

In [ ]:
safe_rets    = pd.DataFrame().reindex_like(risky_rets)
safe_rets[:] = 0.03 / 12

Let us now set our initial account value $A$, i.e., our investment, the floor value, and the multiplier $m$. 

In [ ]:
start_value   = 1000
account_value = start_value
floor         = 0.8
floor_value   = floor * account_value

# recall that a drop > 1/m% would break the floor  
m = 3

In [ ]:
account_history = pd.DataFrame().reindex_like(risky_rets)
cushion_history = pd.DataFrame().reindex_like(risky_rets)
risky_w_history = pd.DataFrame().reindex_like(risky_rets)

Before starting, let us compute the risky wealth series just to make a comparison with the CPPIs that we are going to make below.
That is, let us compute the wealth growths in case we would have only invested in the risky assets. This is simply:

In [ ]:
risky_wealth = start_value * (1 + risky_rets).cumprod()
risky_wealth.head()

Right, run the CPPI strategy:

In [ ]:
# For loop over dates 
for step in range( len(risky_rets.index) ):
    # computing the cushion (as a percentage of the current account value)
    cushion = (account_value - floor_value) / account_value
    
    # compute the weight for the allocation on the risky asset
    risky_w = m * cushion
    risky_w = np.minimum(risky_w, 1)
    risky_w = np.maximum(risky_w, 0)
    # the last two conditions ensure that the risky weight is in [0,1]
    
    # compute the weight for the allocation on the safe asset
    safe_w  = 1 - risky_w
    
    # compute the value allocation
    risky_allocation = risky_w * account_value
    safe_allocation  = safe_w  * account_value
    
    # compute the new account value: this is given by the new values from both the risky and the safe assets
    account_value = risky_allocation * (1 + risky_rets.iloc[step] ) + safe_allocation  * (1 + safe_rets.iloc[step]  )
        
    # save data: current account value, cushions, weights
    account_history.iloc[step] = account_value
    cushion_history.iloc[step] = cushion 
    risky_w_history.iloc[step] = risky_w

# given the CPPI wealth saved in the account_history, we can get back the CPPI returns
cppi_rets = ( account_history / account_history.shift(1) - 1 ).dropna()

Let us look at the account_history created by the CPPI strategy:

In [ ]:
ax = account_history.plot(figsize=(10,5), grid=True)
ax.set_ylabel("wealth ($)")
plt.show()

and let us compare the CPPI strategies with the risky wealths in case of a $100\%$ investment in the risky assets:

In [ ]:
fig, ax = plt.subplots(3,2,figsize=(18,15))
ax = ax.flatten()

# Beer
account_history["Beer"].plot(ax=ax[0], grid=True, label="CPPI Beer")
risky_wealth["Beer"].plot(ax=ax[0], grid=True, label="Beer", style="k:")
ax[0].axhline(y=floor_value, color="r", linestyle="--", label="Fixed floor value")
ax[0].legend(fontsize=11)

# Weights Beer
risky_w_history["Beer"].plot(ax=ax[1], grid=True, label="Risky weight in Beer")
ax[1].legend(fontsize=11)

# Fin
account_history["Fin"].plot(ax=ax[2], grid=True, label="CPPI Fin")
risky_wealth["Fin"].plot(ax=ax[2], grid=True, label="Fin", style="k:")
ax[2].axhline(y=floor_value, color="r", linestyle="--", label="Fixed floor value")
ax[2].legend(fontsize=11)

# Weights Fin
risky_w_history["Fin"].plot(ax=ax[3], grid=True, label="Risky weight in Fin")
ax[3].legend(fontsize=11)

# Steel
account_history["Steel"].plot(ax=ax[4], grid=True, label="CPPI Steel")
risky_wealth["Steel"].plot(ax=ax[4], grid=True, label="Steel", style="k:")
ax[4].axhline(y=floor_value, color="r", linestyle="--", label="Fixed floor value")
ax[4].legend(fontsize=11)

# Weights Steel
risky_w_history["Steel"].plot(ax=ax[5], grid=True, label="Risky weight in Steel")
ax[5].legend(fontsize=11)

plt.show()

As expected, in case of growth periods, the blue line, which is the growth due to the CPPI strategy is normally below the dotted line, which is the growth due to a $100\%$ investment in the risky asset. 

In the case of Beer, we see that from about 2009 the weights of the portfolio becomes $1$ to the risky asset and $0$ to the safe asset, i.e., from 2009 we only invested purely in Beer.

For what concern Finance, the situation is different and we can see the advantage of the CPPI strategy. During the Lehman Brothers crisis, 
a pure investment in Fin would have resulted in a loss that breaks the floor value (note the dotted line going below the red line). On the other hand, we see that the blue line stays above the floor and only from about 2013 the risky weight in Fin becomes $1$ again.

Let us compare statistics of the pure risky asset investment and of the CPPI strategy:

In [ ]:
erk.summary_stats( risky_rets )

In [ ]:
erk.summary_stats( cppi_rets )

If we look at drawdowns, we see that the pure investment in the risky assets would incur in higher losses (drawdowns). 
On the other hand, higher returns are also expected. In fact Beer, for instance, has a annualized return of about $8\%$ versus $7\%$ of the CPPI investment strategy.

### Implementing the drawdown constraint

The above CPPI strategy had a fixed floor value. However, sometimes we would like to **dynamically update the floor value** based on the **previous peak** of the wealth growth, i.e. setting the drawdown constraint.

In case the drawdown is present, the method imposes the multiplier $m=1/\text{drawdown}$. 

For example, let us consider $\text{drawdown}=0.2$ (i.e., $20\%$):

In [ ]:
res = erk.run_cppi(risky_rets, m = 1/0.2, start=1000, floor=0.8, drawdown=0.2, riskfree_rate=0.03)

sector = "Fin"

fig, ax = plt.subplots(1,2,figsize=(18,4))
ax = ax.flatten()

res["Wealth"][sector].plot(ax=ax[0], grid=True, label="CPPI "+sector)
res["Risky Wealth"][sector].plot(ax=ax[0], grid=True, label=sector, style="k:")
res["floor"][sector].plot(ax=ax[0], grid=True, color="r", linestyle="--", label="Fixed floor value")
ax[0].legend(fontsize=11)

# Weights Fin
res["Risky Allocation"][sector].plot(ax=ax[1], grid=True, label="Risky weight in "+sector)
ax[1].legend(fontsize=11)

plt.show()

Compare the statistics of the sector pure risky returns and of the CPPI returns:

In [ ]:
erk.summary_stats(res["Risky Wealth"].pct_change().dropna())

In [ ]:
erk.summary_stats(res["Wealth"].pct_change().dropna())

As expected, for example, for the `Fin` industry, the maximum drawdown taken from the CPPI strategy has been about $19.8\%$, and overall it had an annual return of $6\%$. 

Observe the behaviour of the CPPI strategy when changing the maximum drawdown accepted:

In [ ]:
sector = "Fin"
drawdowns = [0.2, 0.4, 0.6]
    
fig, ax = plt.subplots(1,2,figsize=(18,4))
ax = ax.flatten()

res["Risky Wealth"][sector].plot(ax=ax[0], grid=True, style="k:", label=sector)
ax[0].legend()

summ = pd.DataFrame()
for drawdown in drawdowns:
    res = erk.run_cppi(risky_rets, m = 1/drawdown, start=1000, floor=0.8, drawdown=drawdown, riskfree_rate=0.03)    
    res["Wealth"][sector].plot(ax=ax[0], grid=True, label="CPPI dd={}%, m={}".format(drawdown*100,round(res["m"],1)) )
    res["Risky Allocation"][sector].plot(ax=ax[1], grid=True, label="dd={}%, m={}".format(drawdown*100,round(res["m"],1)) )
    
    summ = pd.concat([summ, erk.summary_stats(res["Wealth"][[sector]].pct_change().dropna())], axis=0)
        
ax[0].legend() 
ax[1].legend(fontsize=11)
ax[1].set_title("Risky weight", fontsize=11)
plt.show()

In [ ]:
summ.index = [["DD20%","DD40%","DD60%"]]
summ

## Random walk generation - Geometric Brownian Motion 

A **Wiener process** $W_{t}$ is a real valued continuous-time **stochastic process** characterised by the following properties:
 1. $W_{0}=0$;
 2. $W$ has independent increments, i.e., for every $t>0$ the future increments $W_{t+u}-W_{t}$ (for $u\geq 0$) 
 are independent of the past values $W_{s}$, for all $s< t$; 
 3. $W$ has Gaussian increments, i.e., $W_{t+u}-W_{t}$ is normally distributed with mean $0$ and variance $u$, $W_{t+u}-W_t \sim{\mathcal {N}}(0,u)$; 
 4. $W$ has continuous paths, i.e., $W_{t}$ is continuous in $t$. 
 5. $W$ is not differentiable

Due to these features, we have that $\mathbb{E}[W_t]=0$, $\text{Var}(W_t) = t$ for a fixed time $t$ and the unconditional 
probability density function is then: 
$$
f_{W_{t}}(x) = \frac{1}{ \sqrt{2\pi t} } e^{\frac{-x^2}{2t}}. 
$$

### Geometric Brownian Motion 

A **stochastic process** $S_t$ is said to follow a **geometric Brownian Motion** (GBM) 
if it satisfies the following **stochastic differential equation** (SDE):
$$
dS_{t} = \mu S_{t}\,dt + \sigma S_t \, dW_{t} 
$$
where $W_{t}$ is a **Brownian motion** (i.e. a Wiener process), 
$\mu$ is called the (percentage) **drift**, and 
$\sigma$ is called the (percentage) **volatility**. Both the drift and the volatility are constants. 
While the drift is used to model a **trend** of the stochastic process, the volatility models unpredictable events 
occurring during this motion (basically, how much the process oscillates around its drift).

#### Returns

In the equation, $dS_t = S_{t+dt} - S_t$ (and so $dW_t = W_{t+dt} - W_t$) for $dt>0$. 
Note that if we divide by $S_t$:
$$
\frac{S_{t+dt} - S_t}{S_t} = \mu dt + \sigma (W_{t+dt} - W_t). 
$$
The quantity on the left-hand side is nothing but that the **percentage return**. 

Also, since $W$ has Gaussian increments $W_{t+dt} - W_t$ 
with zero mean variance $dt$, we can then substitute the increment with a random variable defined as $\sqrt{dt}\xi_t$, 
where $\xi_t \sim \mathcal{N}(0,1)$ for all $t$. 
In fact, $\mathbb{E}[\sqrt{dt}\xi_t] = 0$ and $\text{Var}(\sqrt{dt}\xi_t) = dt\text{Var}(\xi_t)=dt$. 
Therefore, we can rewrite:
$$
\frac{S_{t+dt} - S_t}{S_t} = \mu dt + \sigma \sqrt{dt}\xi_t, 
\quad\xi_t\sim\mathcal{N}(0,1), \; \forall\; t,
$$
and this gives us **a formula to actually generate (percentage) returns** from a geometric brownian motion.
Notice that 
$$
\begin{align}
\mathbb{E}\left[\mu dt + \sigma \sqrt{dt}\xi_t\right] &= \mu dt + \sigma \sqrt{dt} \mathbb{E}[\xi_t] = \mu dt, \\
\text{Var}\left[\mu dt + \sigma \sqrt{dt}\xi_t\right] &= \sigma^2 dt \text{Var}(\xi_t) = \sigma^2 dt,
\end{align}
$$
that is the returns generated in this way have mean $\mu dt$ and volatility $\sigma \sqrt{dt}$.


#### Prices and log-returns

Note that if we divide by $dt$ the stochastic differential equation we have:
$$
\frac{S_{t+dt} - S_t}{dt} = S_t \left( \mu + \sigma \frac{dW_t}{dt} \right), 
$$
which represents the evolution of the process over time. For example, the process $S_t$ can be a **stock price**. 
A solution to the equation can be provided by the following argument.

First of all, we have:
$$
\log(1 + x) \approx x - \frac{x^2}{2},
$$
when $x$ is sufficiently small.
Therefore: 
$$
\begin{equation}
\begin{split}
\log\left( \frac{S_{t+dt}}{S_t} \right) = \log\left(1 + \frac{S_{t+dt} - S_t}{S_t}\right) 
&\approx \frac{S_{t+dt} - S_t}{S_t} - \frac{1}{2}\left( \frac{S_{t+dt} - S_t}{S_t} \right)^2 \\
&\approx \mu dt + \sigma \sqrt{dt}\xi_t - \frac{1}{2} \left( \mu dt + \sigma \sqrt{dt}\xi_t \right)^2 \\
&\approx \mu dt + \sigma \sqrt{dt}\xi_t - \frac{1}{2} \left( \mu^2 \underbrace{dt^2}_{\approx 0} + \sigma^2 dt \underbrace{\xi_t^2}_{\approx \mathbb{E}[\xi_t^2]=1}   + 2\mu \underbrace{dt \sqrt{dt}}_{\approx 0} \sigma \xi_t \right) \\
&\approx \mu dt + \sigma \sqrt{dt}\xi_t - \frac{1}{2} \sigma^2 dt = \left(\mu - \frac{\sigma^2}{2}\right)dt + \sigma\sqrt{dt}\xi_t.
\end{split}
\end{equation}
$$
The quantity on the left-hand side of the equation above is called **log-return** and it follows a dynamic similar to that of the classic percentage return (see above). The difference is that the equation for the og-return a **scaled drift** ($\mu-\sigma^2/2$).

Taking the exponential on both sides:
$$
S_{t+dt} \approx S_t \exp\left( \left(\mu - \frac{\sigma^2}{2}\right)dt + \sigma\sqrt{dt}\xi_t  \right).
$$


To obtain the solution, just note that for every unit $t = ndt$, for a given $n\in\mathbb{N}$, then
$$
\begin{align}
S_t = S_{0+ndt} \approx S_0 \exp\left( \left(\mu - \frac{\sigma^2}{2}\right) (ndt) + \sigma\sqrt{ndt}\xi_t \right)
= S_0 \exp\left( \left(\mu - \frac{\sigma^2}{2}\right)t + \sigma\sqrt{t}\xi_t  \right) 
= S_0 \exp\left( \left(\mu - \frac{\sigma^2}{2}\right)t + \sigma W_t  \right) 
\end{align}
$$
This is the equation for the evolution, for example, of a stock price starting from an initial price of $S_0$. 

Sometimes the **drift** $\mu$ in the geometric Brownian motion is decomposed so that to emphasize the risk-free rate, the volatility, and the sharpe ratio. In fact, recall that the **sharpe ratio** of a stock index was given by
$$
\text{SR} := \lambda = \frac{R_s - R_f}{\sigma} 
\quad\Longrightarrow\quad 
R_s = R_f + \sigma \lambda, 
$$
with $\sigma$ being the volatility of the stock index, $R_f$ being the risk-free rate, and $R_s$ being the annualized return from the stock index. 

From this, we see that since **$\mu$ denotes the expected return of the stock index that we are modelling**, we simply want to emphasize that such **expected return on the risky stock index is given by a risk-free rate $R_f$ plus a risky premium** given by the unit of risk, i.e., the volatility $\sigma$, times the reward per unit per risk, i.e., the sharpe ratio $\lambda$. 

Therefore, we may find the geometric Brownian motion of (percentage) returns with $\mu$ substitued by:
$$
\frac{dS_{t}}{S_t} = (R_f + \sigma \lambda) dt + \sigma dW_{t}. 
$$

#### Interactive random walk simulation

Using **ipywidgets** we can obtain interactive plots. Let us make interactive plots of random walk simulation:

In [ ]:
import ipywidgets as widgets

The function used is called **show_gbm** and basically it simply generate some random walks by calling the **simulate_gbm_from_returns** 
method and plot the random prices. The widget makes the plot interactive:

In [ ]:
def show_gbm(n_scenarios, mu, sigma):
    """
    Draw the results of a stock price evolution under a Geometric Brownian Motion model
    """
    s_0=100
    prices = erk.gbm(n_scenarios=n_scenarios, mu=mu, sigma=sigma, s_0=s_0)
    ax = prices.plot(legend=False, color="indianred", alpha = 0.5, linewidth=2, figsize=(12,5))
    ax.axhline(y=100, ls=":", color="black")
    # draw a dot at the origin
    ax.plot(0,s_0, marker='o',color='darkred', alpha=0.2)

In [ ]:
gbm_controls = widgets.interactive(show_gbm, 
                                   n_scenarios=widgets.IntSlider(min=1, max=1000, step=1, value=1), 
                                   mu=(0., +.2,.01),
                                   sigma=(0, .3, .01))
display(gbm_controls)

The GBM is used to model stock prices in the **Black–Scholes model** and is the most widely used model of stock price behavior.

Some of the arguments for using GBM to model stock prices are the following:

 - The expected returns of GBM are independent of the value of the process (stock price), which agrees with what we would expect in reality;
 - The GBM process only assumes positive values, as real stock prices;
 - The GBM process shows the same kind of *roughness* in its paths as we see in real stock prices;
 - Calculations with GBM processes are relatively easy.
 
However, GBM is not a completely realistic model, in fact: 
 - In real stock prices, volatility changes over time (possibly stochastically), but in GBM processes, volatility is assumed constant;
 - In real life, stock prices often show jumps caused by, for example, news, whereas in GBM processes the path is continuous.

#### Interactive CPPI simulation - Monte Carlo

Here, we called the method **show_cppi** in out kit 
which simulates a CPPI investment strategy starting by returns generated by geometric Brownian Motion. 
We consider only the case of a fixed floor, i.e., the case with no drawdown contraint.

Using the widget interact we make the plot interactive:

In [ ]:
def show_cppi(n_scenarios=50, mu=0.07, sigma=0.15, m=3, floor=0., riskfree_rate=0.03, steps_per_year=12, y_max=100):
    """
    Plot the results of a Monte Carlo Simulation of CPPI
    """
    start = 100
    sim_rets = erk.gbm(n_scenarios=n_scenarios, mu=mu, sigma=sigma, prices=False, steps_per_year=steps_per_year)
    risky_r = pd.DataFrame(sim_rets)
    # run the "back"-test
    btr = erk.run_cppi(risky_r=pd.DataFrame(risky_r),riskfree_rate=riskfree_rate,m=m, start=start, floor=floor)
    wealth = btr["Wealth"]

    # calculate terminal wealth stats
    y_max=wealth.values.max()*y_max/100
    terminal_wealth = wealth.iloc[-1]
    
    tw_mean = terminal_wealth.mean()
    tw_median = terminal_wealth.median()
    failure_mask = np.less(terminal_wealth, start*floor)
    n_failures = failure_mask.sum()
    p_fail = n_failures/n_scenarios

    e_shortfall = np.dot(tv-start*floor, failure_mask)/n_failures if n_failures > 0 else 0.0

    # Plot!
    fig, (wealth_ax, hist_ax) = plt.subplots(nrows=1, ncols=2, sharey=True, gridspec_kw={'width_ratios':[3,2]}, figsize=(24, 9))
    plt.subplots_adjust(wspace=0.0)
    
    wealth.plot(ax=wealth_ax, legend=False, alpha=0.3, color="indianred")
    wealth_ax.axhline(y=start, ls=":", color="black")
    wealth_ax.axhline(y=start*floor, ls="--", color="red")
    wealth_ax.set_ylim(top=y_max)
    
    terminal_wealth.plot.hist(ax=hist_ax, bins=50, ec='w', fc='indianred', orientation='horizontal')
    hist_ax.axhline(y=start, ls=":", color="black")
    hist_ax.axhline(y=tw_mean, ls=":", color="blue")
    hist_ax.axhline(y=tw_median, ls=":", color="purple")
    hist_ax.annotate(f"Mean: ${int(tw_mean)}", xy=(.7, .9),xycoords='axes fraction', fontsize=24)
    hist_ax.annotate(f"Median: ${int(tw_median)}", xy=(.7, .85),xycoords='axes fraction', fontsize=24)
    if (floor > 0.01):
        hist_ax.axhline(y=start*floor, ls="--", color="red", linewidth=3)
        hist_ax.annotate(f"Violations: {n_failures} ({p_fail*100:2.2f}%)\nE(shortfall)=${e_shortfall:2.2f}", xy=(.7, .7), xycoords='axes fraction', fontsize=24)

cppi_controls = widgets.interactive(show_cppi,
                                   n_scenarios=widgets.IntSlider(min=1, max=1000, step=5, value=50), 
                                   mu=(0., +.2, .01),
                                   sigma=(0, .3, .05),
                                   floor=(0, 2, .1),
                                   m=(1, 5, .5),
                                   riskfree_rate=(0, .05, .01),
                                   steps_per_year=widgets.IntSlider(min=1, max=12, step=1, value=12,
                                                          description="Rebals/Year"),
                                   y_max=widgets.IntSlider(min=0, max=100, step=1, value=100,
                                                          description="Zoom Y Axis")
)
display(cppi_controls)


---

# Chapter 4: Introduction to Asset-Liability Management

---


# Chapter 4: Introduction to Asset-Liability Management

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import seaborn as sns
import edhec_risk_kit as erk

# using seaborn style (type plt.style.available to see available styles)
plt.style.use("seaborn-pastel") 

%load_ext autoreload
%autoreload 2
%matplotlib inline

## Time value of money


### Present and future value

In general, we say that $P_{t+k}$ is the **future value (FV)** of the amount of money equal to $P_t$. Likewise, we say that 
$P_t$ is the **present value (PV)** of the amount of money $P_{t+k}$ due in the future (at time $t+k$).

$$
FV = PV(1 + R)^k
\qquad\text{and}\qquad
PV = \frac{FV}{(1 + R)^k}.
$$


### Discount rate

The factor
$$
\frac{1}{(1+R)^k}, 
$$
is called the **discount factor**.


### Cumulative present value (of future cash flows)

The **cumulative present value** (PV) of future cash flows can be calculated by summing the contributions of future values $FV_t$, i.e., the value of cash flow at time $t$. 


In general, if we **denote the future cash flows as the future values**, we have that the present value of 
a **liability** $L$ is:
$$
PV(L) = \sum_{i=1}^N \frac{FV_{t_i}}{(1+R)^{t_i}} := \sum_{i=1}^N B(t_i) L_{t_i}, 
$$
and $L_{t_i}$ denotes the future cash flow at time $t_i$, i.e., the **liability at time $t_i$**, and 
$B(t_i) := 1 / (1+R)^{t_i}$ denotes the **discount rate**.

### Funding ratio

The funding ratio is therefore a simply ratio between the value of assets I currently hold and the present value of our liability:
$$
FR = \frac{\text{assets values}}{{PV(L)}}.
$$
If the value of my current assets is bigger than $PV(L)$, then $FR >1$ and this means that I have enough funds to pay of my liability in the coming future. In the example above:

In [ ]:
# Example calculating discounted value
erk.discount_simple(10, .03)

In [ ]:
# Example calculating present value
liabilities = pd.Series(data=[1, 1.5, 2, 2.5], index=[3, 3.5, 4, 4.5])
erk.pv_simple(liabilities, 0.03)

We can now compute the funding ratio, based on current asset values:

In [ ]:
# Example calculating the funding ration
erk.funding_ratio_simple(5, liabilities, 0.03)

Thus, in order to be able to meet the future liabilities as specified in `liabilities` at the indicated future times, the present funding ratio should be 80.2% (6.23 million in current assets in order to pay 7 million at years 3, 3.5, 4, and 4.5).

As the widget below shows, even if your assets do not go down in value, cash can be a risky asset if you think about the funding ratio rather than the asset value. Even though cash is a "safe asset" in the sense that the asset value does not go down, cash can be a very risky asset because the value of the liabilities goes up when interest rates go down. Therefore, if you think about your savings in terms of funding ratio (i.e. how much money do you have compared to what you need) then cash is a risky asset and can result in a decline in your funding ratio.

In [ ]:
def show_funding_ratio(assets, r):
    fr = erk.funding_ratio_simple(assets, liabilities, r)
    print(f'{fr*100:.2f}%')
    
controls = widgets.interactive(show_funding_ratio,
                                   assets=widgets.IntSlider(min=1, max=10, step=1, value=5),
                                   r=(0, .20, .01)
)
display(controls)

### Nominal rate and Effective annual interest rate


#### Short-rate v.s. Long-Rate (annualized)

In general, given a **nominal interest rate** $r$ (also called **instantaneous interest rate**) and $N$ **number of periods**, i.e., the number of payments (of the investment, loan, and so on), 
the total return is given by:
$$
R = \left(1 + \frac{r}{N}\right)^N - 1.
$$
Such $R$ is nothing but that the **annualized return** (as we used to call it so far) or the **effective annual interest rate** 
obtained from **discrete compounding**. 

Note that, from formula above, we also have:
$$
r = N((1+R)^{1/N}-1).
$$

In [ ]:
# Consider a nominal interest rate of 10% and monthly payments
nominal_rate = 0.1
periods_per_year = 12

# that is, every month, we have the following rates
rets = pd.DataFrame( [nominal_rate/periods_per_year for i in range(10)] )
rets.head(3)

Due to compounding, we have:

In [ ]:
ann_ret = erk.annualize_rets(rets, periods_per_year)[0]
ann_ret

In [ ]:
R = (1 + nominal_rate / periods_per_year)**periods_per_year - 1
R

Notice that we got, obviously, the same number (and that it does not depend on the number of returns in the dataframe...). 

#### Continuous compounding
By having the annualized return expressed in the form above, we realize that when $N$ becomes large we have:
$$
\lim_{N\to \infty} \left(1 + \frac{r}{N} \right)^N = e^{r}, 
$$
and then 
$$
R = \lim_{N\to \infty} \left(1 + \frac{r}{N} \right)^N  - 1 \approx e^r - 1
\quad\text{from which}\quad 
r \approx \log{(1 + R)}.
$$
This is the case of continuosly compounded returns. Of course it is a theoretical way of compounding although in some cases it can effectively be used. 

## CIR model: simulate changes in interest rates

The **CIR model** (from **Cox, Ingersoll, Ross**) is used to simulate changes in interest rates (returns) and it is an extension of the **Vasicek** model to prevent negative interest rates. It is a type of **one factor model**, or **short-rate model**, 
as it describes interest rate movements as driven by only one source of market risk. 
The model can be used in the valuation of interest rate derivatives. 

The dynamic for interest rates is modelled in the following way:
$$
dr_t = a(b-r_t)dt + \sigma\sqrt{r_t}dW_t,
$$
where, $W_t$ is a Brownian motion (see Week 3) which models the random market risk factor, $b$ is a **(long-term) mean interest rate**, and the difference $b-r_t$ denotes how far away is the current interest rate from the (long-term) mean interest rate. 
Finally, $a$ is the **mean-reversion speed**, or speed of adjustment to the mean, which models **how fast** do we revert to the (long-term) mean interest rate.

The movements of the interest rates then depend on the long-term mean rate ($b$) and on 
how fast the dynamic changes and try to get close to the mean rate (by $a$) after deviations 
(by the noise given by $\sigma\sqrt{r_t}dW_t$). 
Note that the standard deviation factor $\sigma\sqrt{r_t}$ avoids the possibility of negative interest rates for all positive values of $a$ and $b$. Moreover, an interest rate equal to zero is also precluded if $2ab\geq \sigma^2$. 

More generally, when the rate $r_t$ is close to zero, the standard deviation $\sigma\sqrt{r_t}$ also becomes very small, which dampens the effect of the random shock on the rate. Consequently, in this case, the evolution of the rate becomes dominated by the drift factor 
which pushes the rate upwards (towards equilibrium).

In [ ]:
def show_cir(r_0=0.03, a=0.5, b=0.03, sigma=0.05, n_scenarios=5):
    rates, prices = erk.cir(r_0=r_0, a=a, b=b, sigma=sigma, n_scenarios=n_scenarios)
    rates.plot(legend=False, figsize=(12,5))
    
import ipywidgets as widgets
from IPython.display import display
    
controls = widgets.interactive(show_cir,
                              r_0 = (0, .15, .01),
                              a = (0, 1, .1),
                               b = (0, .15, .01),
                               sigma= (0, .1, .01),
                               n_scenarios = (1, 100))
display(controls)

### Using the CIR model for zero-coupon bond pricing

Under the **no-arbitrage assumption**, a **zero-coupon bond** may be priced using the interest rate process modeled by the CIR model. 
The bond price $P(t,T)$ of a zero-coupon bond with maturity $T$ is exponential affine in the interest rate and is given by:
$$
P(t,T) = A(t,T)e^{-B(t,T)r_t}, 
$$
where
$$
\begin{align}
A(t,T) &:= \left( \frac{ 2h e^{(a+h)\tau/2}  }{ 2h+(a+h)(e^{h\tau}-1) }   \right)^{2ab/\sigma^2}, \\
B(t,T) &:= \frac{ 2(e^{h\tau} - 1)  }{ 2h+(a+h)(e^{h\tau}-1) },  \\
h &:= \sqrt{a^2 + 2\sigma^2}, \\
\tau &:= T - \tau.
\end{align}
$$

In [ ]:
def show_cir_prices(r_0=0.03, a=0.5, b=0.03, sigma=0.05, n_scenarios=5):
    rates, prices = erk.cir(r_0=r_0, a=a, b=b, sigma=sigma, n_scenarios=n_scenarios)
    prices.plot(legend=False, figsize=(12,5))

controls = widgets.interactive(show_cir_prices,
                              r_0 = (0, .15, .01),
                              a = (0, 1, .1),
                               b = (0, .15, .01),
                               sigma= (0, .1, .01),
                               n_scenarios = (1, 100))
display(controls)

## Liability hedging 

Since we have a model to simulate interest rates and changes in the price of **zero-coupon bonds** 
we can see what happens to the hedging strategy **in case we use zero-coupon bonds as hedge instead of cash**. 

We have seen that changes in interest rates has enormous impact on the liabilities we may have for the next future, and therefore, on the funding ratios. Hence it makes sense to make sure that we know how the portfolio is going to behave as interest rates 
fluctuate over time.

The problem is: **we have some liability to satisfy in the future**, and as long as interest rates changes over times, we have to make sure to meet such a liability by using **a hedging strategy so that an increase in value of our asset will be enough**. 

Suppose we have an initial amount of $0.75$ million dollars (**asset_0**) and that we have a total liability of $1$ million dollars (**tot_liab**) due in $10$ years. Also, suppose that the mean rate (the nominal rate of this liability) is $3\%$. 

In [ ]:
asset_0  = 0.75
tot_liab = 1

mean_rate = 0.03
n_years   = 10
n_scenarios = 10
periods_per_year = 12

#### Simulate interest rates and zero-coupon bond prices

Let us simulate interest rates (starting from the mean rate) for the next 10 years:

In [ ]:
rates, zcb_price = erk.cir(n_years = n_years, n_scenarios=n_scenarios, a=0.05, b=mean_rate, sigma=0.05, steps_per_year=periods_per_year, r_0=None)
rates.head()

Furthermore, let us suppose that **our liabilities over time are given by the zero-coupon bond prices** that we have just simulated. 
This is going to be simply a trick to have changes in the liability according to the change in interest rates:

In [ ]:
L = zcb_price
L.head()

#### Hedging by buying zero-coupon bonds

Our task is meet the liability in 10 years. **The idea is to invest our current asset in a zero-coupon bond**. 

We know what is the price of a zero-coupon bond that matures in 10 years. Since, by definition, **such bond does not pay coupons and simply returns at maturity the face value plus the given interest**, its price is simply given by the present value of this last amount:

In [ ]:
# price of a zcb with maturity 10 years and rate equal to the mean rate
zcb = pd.DataFrame(data=[tot_liab], index=[n_years])
zcb_price_0 = erk.pv_simple( zcb, mean_rate )
zcb_price_0

i.e., a zero-coupon bond paying off $1$ million dollars plus an interest of $3\%$ in 10 years is worth about $0.74$ million dollars today. 

Hence, **suppose we invest into such zero-coupon bond**. 
First of all, given our initial asset, we can buy today the following number of bonds:

In [ ]:
n_bonds = float(asset_0 / zcb_price_0)
n_bonds

so basically, $1$ such bond, and since we know how interest rates will change over time, 
we also know **the future value of our asset that have been invested in this zero-coupon bond**. 
In fact, we know how many bonds we buy and how the price will change. Then:

In [ ]:
asset_value_of_zcb = n_bonds * zcb_price
asset_value_of_zcb.head()

#### Hedging by investing in cash 
Suppose now that instead of putting our money in zero-coupon bonds **we just put our money in cash**. 
What is the future value of our asset then? 

Since interest changes over time, the future value of our asset is simply obtained by compounding with respect to the interest rates, i.e.,
$$
\text{asset}_{10y} = \text{asset}_0 (1+r_{0,1})(1+r_{1,2})\cdots.
$$

In [ ]:
# we have to divide by periods_per_year since rates are returned as annual rates
asset_value_in_cash = asset_0 * (1 + rates/periods_per_year).cumprod()
asset_value_in_cash.head()

#### Result: comparing the two investments

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(20,5))

asset_value_in_cash.plot(ax=ax[0], grid=True, legend=False, color="indianred", title="Future value of asset put in cash")
asset_value_of_zcb.plot(ax=ax[1], grid=True, legend=False, color="indianred", title="Future value of asset put in ZCB")
ax[0].axhline(y=1.0, linestyle=":", color="black")
ax[1].axhline(y=1.0, linestyle=":", color="black")
ax[0].set_ylabel("millions $")
ax[1].set_ylabel("millions $")
if periods_per_year == 12:
    ax[0].set_xlabel("months ({:.0f} years)".format((len(asset_value_in_cash.index)-1)/periods_per_year))
    ax[1].set_xlabel("months ({:.0f} years)".format((len(asset_value_in_cash.index)-1)/periods_per_year))

plt.show()

We see that, although the increase in value in the left-hand side plot is smoother, there have been **some scenarios in which the investment of our asset value in cash did not provide the $1$ million dollars of liability** that we have to pay.

On the other hand, notheless the oscillation in value, the investment in **the zero-coupon bond always guarantee the money we need at the maturity**.

Let us look at the **funding ratios** of the two investments, that is, the ratio between the asset value and the liability:

In [ ]:
fr_cash = asset_value_in_cash / L
fr_zcb  = asset_value_of_zcb  / L

fig, ax = plt.subplots(2,2,figsize=(20,8))

fr_cash.plot(ax=ax[0,0], grid=True, legend=False, color="indianred", 
             title="Funding ratios of investment in cash ({} scenarios)".format(n_scenarios))
fr_zcb.plot(ax=ax[0,1], grid=True, legend=False, color="indianred", 
            title="Funding ratios of investment in ZCB ({} scenarios)".format(n_scenarios))

ax[0,0].axhline(y=1.0, linestyle=":", color="black")
ax[0,1].axhline(y=1.0, linestyle=":", color="black")

fr_cash.pct_change().plot(ax=ax[1,0], grid=True, legend=False, color="indianred",
                          title="Pct changes in funding ratios of investment in cash ({} scenarios)".format(n_scenarios))
fr_zcb.pct_change().plot(ax=ax[1,1], grid=True, legend=False, color="indianred", 
                         title="Pct changes in funding ratios of investment in ZCB ({} scenarios)".format(n_scenarios))
plt.show()

Let us now look at the **terminal** funding ratios, i.e., the funding ratio when maturity is met. Since we wan to plot an histogram for the distribution of the funding ratios, we repeat all the steps above for a larger number of scenarios:

In [ ]:
# simulate more scenarios
n_scenarios = 5000
rates, zcb_price = erk.cir(r_0=0.03, b=0.03, n_scenarios=10000)
# assign liabilities
L = zcb_price
# ZCB investment:
zcb = pd.DataFrame(data=[tot_liab], index=[n_years])
zcb_price_0 = erk.pv_simple( zcb, mean_rate )
n_bonds = float(asset_0 / zcb_price_0)
asset_value_of_zcb = n_bonds * zcb_price
# Cash investment 
asset_value_in_cash = asset_0 * (1 + rates/periods_per_year).cumprod()

# terminal funding ratios
terminal_fr_zcb  = asset_value_of_zcb.iloc[-1]  / L.iloc[-1]
terminal_fr_cash = asset_value_in_cash.iloc[-1] / L.iloc[-1]

ax = terminal_fr_cash.plot.hist(label="(Terminal) Funding Ratio of investment in cash", bins=50, figsize=(12,5), color="orange", legend=True)
terminal_fr_zcb.plot.hist(ax=ax, grid=True, label="(Terminal) Funding Ratio of investment in ZCB", bins=50, legend=True, color="blue", secondary_y=True)
ax.axvline(x=1.0, linestyle=":", color="k")
ax.set_xlabel("funding ratios")
plt.show()

This shows that although there are many scenarios in which we can do very well with cash, i.e., our investment in cash will gain enough value to pay the liability (funding ratios larger than $1$), **there are also many scenarios for which the funding ratio will be less than $1$**. 
On the other hand, the terminal funding ratio for the investment of zero-coupon bond will be always larger than or equal to $1$.

## Coupon-bearing bonds 

$$
\text{Bond price}= PV = \sum_{i=1}^N \left(\frac{C_{t_i}}{(1+\text{YTM})^{t_i}}\right)  +  \frac{F}{(1+\text{YTM})^{t_N}} 
= C\left(\frac{1-(1+\text{YTM})^{-t_N}}{\text{YTM}}\right) +  \frac{F}{(1+\text{YTM})^{t_N}},
$$ 
where $N$ is the number of coupons paid, 
$C_{t_i}$ is the coupon paid at time $t_i$ (they are all the same), $F$ is the face value of the bond, 
and $\text{YTM}$ is the **yield to maturity**.

For example, let us see the cash flow generated by a bond:

In [ ]:
principal        = 100 
maturity         = 3
ytm              = 0.05
coupon_rate      = 0.03 
coupons_per_year = 2

In [ ]:
cf = erk.bond_cash_flows(principal=principal, maturity=maturity, coupon_rate=coupon_rate, coupons_per_year=coupons_per_year)
cf

Since the bond pays two coupons per year, the yield of each payment is $0.015$ (half of the coupon rate), and there a total of $6$ payments where the last one (at maturity) consists of the face value plus the last coupon. 

Look at the price of such bond:

In [ ]:
bond_price = erk.bond_price(principal=principal, maturity=maturity, coupon_rate=coupon_rate, coupons_per_year=coupons_per_year, discount_rate=ytm)
bond_price

In [ ]:
# total sum paid by the bond if held until maturity 
tot_bond_paym = cf.sum()
tot_bond_paym

In [ ]:
# gain (in money) of the investment in such bond
gain = -bond_price + tot_bond_paym
gain

Basically, the $\text{YTM}=0.05$ is **approximately** given by the annual rate which (after compounding) returns a total amount equal 
to **tot_bond_paym** by having investing an initial amount equal to the bond price **bond_price**, i.e, 

In [ ]:
# tot_bond_paym = bond_price * (1 + r)**maturity, hence 
r = (tot_bond_paym / bond_price )**(1/maturity) - 1
r

### Relation between yield to maturity and bond price

When the price of the bond is less than its face value (par value), the bond is selling **at a discount**. This happens **when the ytm is larger than the coupon rate**.

Conversely, if the price of the bond is greater than its face value, the bond is selling **at a premium**. This happens **when the ytm is smaller than the coupon rate**.

If the price of the bond is equal to its face value, the bond is selling **at par**. This happens **when the ytm is equal to the coupon rate**.

Therefore, **the ytm and the bond price is inversely correlated**:

In [ ]:
coupon_rate = 0.04
principal = 100
ytm = np.linspace(0.01, 0.10, 20)
bond_prices = [erk.bond_price(maturity=3, principal=principal, coupon_rate=coupon_rate, coupons_per_year=2, discount_rate=r) for r in ytm]

ax = pd.DataFrame(bond_prices, index=ytm).plot(grid=True, title="Relation between bond price and YTM", figsize=(9,4), legend=False)
ax.axvline(x=coupon_rate, linestyle=":", color="black")
ax.axhline(y=principal, linestyle=":", color="black")
ax.set_xlabel("YTM")
ax.set_ylabel("Bond price (Face value)")
plt.show()

### Changes in price 
Note that the yield to maturity is a variable that changes over time. 
In fact, it is an interest rate which changes and, in turn, would change the price of the bond. 

Let us see how the price of a coupon-bearing bond changes as interest rates (i.e., yields to maturity) change. 
First of all, we simulate the interest rates usig the CIR model:

In [ ]:
n_years          = 10
n_scenarios      = 10
b                = 0.03
periods_per_year = 2

rates, _ = erk.cir(n_years = n_years,
                   n_scenarios = n_scenarios, 
                   a=0.02, b=b, sigma=0.02, 
                   steps_per_year=periods_per_year, r_0=None)

rates.tail()

In [ ]:
# bond features
principal        = 100
maturity         = n_years
coupon_rate      = 0.04
coupons_per_year = periods_per_year

bond_prices = erk.bond_price(principal=principal, maturity=maturity, coupon_rate=coupon_rate, 
                             coupons_per_year=coupons_per_year, discount_rate=rates) 

fig, ax = plt.subplots(1,2,figsize=(20,5))
rates.plot(ax=ax[0], grid=True, legend=False) 
bond_prices.plot(ax=ax[1], grid=True, legend=False)
ax[0].set_xlabel("months")
ax[0].set_ylabel("interest rate (ytms)")
ax[1].set_xlabel("months")
ax[1].set_ylabel("bond price")
plt.show()

On the left, we see how interest rates changes over time, and on the right, we see how, in turn, the bond price changes as well.
Also notice that, the price of the bonds at maturity is the same in all scenarios as it is given by the principal plus the coupon-interest. 

## Macaulay Duration 

The **Macaulay duration** is the **weighted average maturity of cash flows**. Suppose the case of a bond. We have a set 
of fixed cash flows (the coupon payments and the last payment given by the principal) whose total present value is given by
$$
PV = \sum _{i=1}^{N} PV_{i}.
$$
Hence, the Macaulay duration is defined as
$$
\text{MacD} 
:= \frac{ \sum_{i=1}^N t_i PV_i}{ \sum _{i=1}^{N} PV_{i} } 
= \frac{ \sum_{i=1}^N t_i PV_i}{PV} 
= \sum_{i=1}^N w_i t_i
\quad\text{with}\quad
w_i := \frac{PV_i}{PV}, 
$$
where $PV_{i}$ is the present value of the cash flow paid at time $t_i$. 
Note that **the Macaulay duration of a zero-coupon bond is equal to the maturity** as such a bond does not pay regular coupons 
but only one single payment at maturity.

In [ ]:
principal        = 1000
maturity         = 3
ytm              = 0.06
coupon_rate      = 0.06
coupons_per_year = 2
cf = erk.bond_cash_flows(principal=principal, maturity=maturity, coupon_rate=coupon_rate, coupons_per_year=coupons_per_year)
cf

In [ ]:
# the discount rate is the YTM divided by the coupons_per_year
macd = erk.macaulay_duration(cf, discount_rate=ytm/coupons_per_year) 
macd = macd / coupons_per_year
macd

that is, the duration of the bond is about $2.79$ years versus the real maturity of $3$ years. **Note that we have divided by the number of coupons per year** because the returned duration was computed with dates from 1 to 6, hence it was about $5.57$ 
(this is the duration expressed in the number of total *periods*). 

Another way to compute the duration would be to **normalize the cash flows dates**:

In [ ]:
cf = erk.bond_cash_flows(principal=principal, maturity=maturity, coupon_rate=coupon_rate, coupons_per_year=coupons_per_year)
cf.index = cf.index / coupons_per_year
cf

In [ ]:
# and now the discount rate is simply the YTM
erk.macaulay_duration(cf, discount_rate=ytm)

and note that in this case we use **only the YTM as discount rate**.

The interpretation is simply that **since the bond is paying coupons during its life, the time waited to get back the money is, effectively, less than the maturity**, just because we already receive some money during the life of the bond.

Let us verify that the duration of a zero-coupon bond is equal to the maturity, no matters the interest rate:

In [ ]:
# ZCB: only one flux at maturity
maturity = 3
cf = pd.DataFrame(data=[100], index=[maturity])
macd = erk.macaulay_duration(cf, discount_rate=0.05) # it does not matter the rate here
macd

## Liability Driven Investing (LDI)



### Duration Matching

Assume we have a liability of 100K in 10 years time and another of 100K in 12 years time. Assume interest rates are 4%. What is the duration of the liabilities?

In [ ]:
liabilities = pd.Series(data = [100000, 100000], index=[10, 12])
erk.macaulay_duration(liabilities, .04)

Now assume we have two types of bonds available. We have a 10 year bond and a 15 year bond. Each of them pays a 5% coupon once a year and has a face value of \\$1000. What are the durations of these bonds?

In [ ]:
md_10 = erk.macaulay_duration(erk.bond_cash_flows(10, 1000, .05, 1), .04)
print(md_10)
md_20 = erk.macaulay_duration(erk.bond_cash_flows(20, 1000, .05, 1), .04)
print(md_20)

Therefore, we need to hold a portfolio of these two bonds that has a combined target duration that matches the duration of the liability, which is given by the following expression, where $w_s$ is the weight in the short duration bond whcih has duration $d_s$ and the duration of the longer bond is $d_l$. We designate the targeted duration as $d_t$.

In our case, the fraction in the short duration asset $w_s$ should be such that:

$$ w_s \times 8.19 + (1-w_s) \times 13.54 = 10.96 $$

more generally:

$$ w_s \times d_s + (1-w_s) \times d_l = d_t $$

rearranging gives:

$$ w_s = \frac{d_l -d_t}{d_l - d_s} $$

The function `match_duration` in the `edhc_risk_kit` will do this for us.

In [ ]:
# Example
short_bond = erk.bond_cash_flows(10, 1000, .05, 1)
long_bond = erk.bond_cash_flows(20, 1000, .05, 1)
w_s = erk.match_durations(liabilities, short_bond, long_bond, 0.04)
w_s # the weight for the short bond

In [ ]:
p_short = erk.bond_price(10, 1000, .05, 1, 0.04)
p_long = erk.bond_price(20, 1000, .05, 1, 0.04)
a_0 = 130000
dm_assets=pd.concat([a_0*w_s*short_bond/float(p_short),a_0*(1-w_s)*long_bond/float(p_long)])
erk.macaulay_duration(dm_assets, 0.04)

In [ ]:
cfr = erk.funding_ratio(dm_assets, liabilities, 0.04)
cfr

In [ ]:
lb_assets = a_0*long_bond/float(p_long)
sb_assets = a_0*short_bond/float(p_short)

In [ ]:
rates = np.linspace(0, .1, 20)
fr_change = pd.DataFrame({
    "Long Bond":[erk.funding_ratio(lb_assets, liabilities, r) for r in rates],
    "Short Bond":[erk.funding_ratio(sb_assets, liabilities, r) for r in rates],
    "Duration Matched Bonds":[erk.funding_ratio(dm_assets, liabilities, r) for r in rates]
}, index=rates)
fr_change.plot(title='Change in Funding Ratio', figsize=(10,6));

# Monte Carlo Simulation of Prices of Coupon-Bearing Bond using CIR


Note that when interest rates rise, it is a fallacy that holding an allocation to bonds will give you the benefit of that increase. In fact, the opposite happens since the price of the bond will fall, and as a result your account still suffer a capital loss.

Let's examine what happens to your wealth when you use a portfolio of stocks and bonds over a 5 year period.

For simplicity, we'll assume you are holding a bond that has a maturity of 5 years and for simplicity with avoiding intra-coupon caclulations, let's assume it pays a 5% coupon and the coupon is paid out each month and interest rates change from 3% to 3.2%


In [ ]:
erk.bond_price(5,100,.05,12,.03)

In [ ]:
erk.bond_price(5,100,.05,12,.032)

We'll generate interest rates using the CIR model:

In [ ]:
rates, zc_prices = erk.cir(10, 500, b=0.03, r_0 = 0.03)

When we start out, all rates are at 3% and so the prices are all the same:

In [ ]:
erk.bond_price(5,100,.05,12, rates.iloc[0][[1,2,3]])

In [ ]:
rates[[1,2,3]].head()

At t=0 interest rates are the same across all scenarios. However at the first step, we see that (i) the maturity decreases and (ii) interest rates change, and so the bond prices diverge:

In [ ]:
erk.bond_price(5,100,.05,12, rates.iloc[1][[1,2,3]])

### Simulating Prices of a Coupon-bearing Bond

From the ability to compute bond prices from rates, we can stitch together the prices of a bond over time:

In [ ]:
prices = erk.bond_price(10, 100, .05, 12, rates[[1,2,3,4]])
prices.plot(figsize=(12,6));

In [ ]:
prices.head()

Because the interest rate at the time we bought the bond was less than the coupon rate, we bought it at a premium, taking a capital loss when we sold it at the end. We need to compute the Total Return of a bond, which is the price return PLUS the dividend with the function `bond_total_return`.

In [ ]:
p = erk.bond_price(10, 100, .05, 12, rates[[1,2,3,4]])
btr = erk.bond_total_return(p, 100, .05, 12)
erk.annualize_rets(btr, 12)

Which gives us the approximately 3% return we expected, because that was the prevailing rate when we bought it. We also assumed that we reinvested the coupon in the bond and that is why we did not get the exact same return in each case, since we would observe prices based on the then-prevalent interest rates.

## Putting it all together: Monte Carlo Simulation of Asset Allocation

Now that we have a way to generate prices from which to derive returns, we can experiment with allocating across the different Asset Classes.

Let's start by examining the performance of a 70-30 allocation to Stocks and Bonds. Assume Stocks return an average of 7% per year with a 15% vol and use the CIR model to generate bond prices for a 10 year and 30 year bond that pays a 5% coupon. For simplicity, assume the coupon is paid monthly to avoid having to deal with partial coupons. Assume the Bond Portfolio consists of 60% in the 10 year bond and 40% in the 30 year bond.

In [ ]:
price_10 = erk.bond_price(10, 100, .05, 12, rates)
price_10[[1,2,3]].tail()

In [ ]:
price_30 = erk.bond_price(30, 100, .05, 12, rates)
price_30[[1,2,3]].tail()

In [ ]:
rets_30 = erk.bond_total_return(price_30, 100, .05, 12)
rets_10 = erk.bond_total_return(price_10, 100, .05, 12)

Now we can assume monthly rebalancing and compute the monthly returns of the bond portfolio:

In [ ]:
rets_bonds = .6*rets_10 + .4*rets_30
mean_rets_bonds = rets_bonds.mean(axis='columns')
erk.summary_stats(pd.DataFrame(mean_rets_bonds))

In [ ]:
price_eq = erk.gbm(n_years=10,n_scenarios=500,mu=0.07, sigma=0.15)
price_eq.shape

In [ ]:
rets_eq = price_eq.pct_change().dropna()
rets_eq.shape

In [ ]:
rets_bonds.shape

Now we assume a 70/30 Equity/Debt portfolio where the equity is simulated by geometric brownian motion and the debt portfolio is the 60/40 short-long bond portfolio simulated using CIR model:

In [ ]:
rets = .70*rets_eq + 0.3*rets_bonds
rets_mean = rets.mean(axis='columns')
erk.summary_stats(pd.DataFrame(rets_mean))

## Naive Risk Budgeting Strategies between PSP and GHP


### The Simplest Allocator - Fixed Mix

The allocator's job is to come up with a time series of weights, so let's create the simplest possible allocator - one that puts a fixed fraction in the first portfolio and the remaining in the second


```python

def bt_mix(r1, r2, allocator, **kwargs):
    """
    Runs a back test (simulation) of allocating between a two sets of returns
    r1 and r2 are T x N DataFrames or returns where T is the time step index and N is the number of scenarios.
    allocator is a function that takes two sets of returns and allocator specific parameters, and produces
    an allocation to the first portfolio (the rest of the money is invested in the GHP) as a T x 1 DataFrame
    Returns a T x N DataFrame of the resulting N portfolio scenarios
    """
    if not r1.shape == r2.shape:
        raise ValueError("r1 and r2 should have the same shape")
    weights = allocator(r1, r2, **kwargs)
    if not weights.shape == r1.shape:
        raise ValueError("Allocator returned weights with a different shape than the returns")
    r_mix = weights*r1 + (1-weights)*r2
    return r_mix

def fixedmix_allocator(r1, r2, w1, **kwargs):
    """
    Produces a time series over T steps of allocations between the PSP and GHP across N scenarios
    PSP and GHP are T x N DataFrames that represent the returns of the PSP and GHP such that:
     each column is a scenario
     each row is the price for a timestep
    Returns an T x N DataFrame of PSP Weights
    """
    return pd.DataFrame(data = w1, index=r1.index, columns=r1.columns)
```

We are now ready to rerun the experiment we ran last time ... a bond portfolio of 60% in the 10 year bond and 40% in the 30 year bond to generate a fixed mix bond portfolio.

In [ ]:
rates, zc_prices = erk.cir(10, 500, b=0.03, r_0 = 0.03)
price_10 = erk.bond_price(10, 100, .05, 12, rates)
price_30 = erk.bond_price(30, 100, .05, 12, rates)
rets_30 = erk.bond_total_return(price_30, 100, .05, 12)
rets_10 = erk.bond_total_return(price_10, 100, .05, 12)
rets_bonds = erk.bt_mix(rets_10, rets_30, allocator=erk.fixedmix_allocator, w1=.6)
mean_rets_bonds = rets_bonds.mean(axis='columns')
erk.summary_stats(pd.DataFrame(mean_rets_bonds))

Next, we'll use this to create a 70-30 Stock Bond Mix. First, we'll generate stock returns:

In [ ]:
price_eq = erk.gbm(n_years=10,n_scenarios=500,mu=0.07, sigma=0.15)
rets_eq = price_eq.pct_change().dropna()
rets_zc = zc_prices.pct_change().dropna()

And next, we'll use the mix backtester to build a 70-30 Stock-Bond mix. One way to assess the performance is, as before, to generate a composite and produce summary stats on the composite. We'll also examine a second approach, which is to compute summary stats on each scenario and average the summary stats.

In [ ]:
rets_7030b = erk.bt_mix(rets_eq, rets_bonds, allocator=erk.fixedmix_allocator, w1=0.7)
rets_7030b_mean = rets_7030b.mean(axis='columns')
erk.summary_stats(pd.DataFrame(rets_7030b_mean))

In [ ]:
# Approach 2: compute stats on each scenario and then average
summaries = erk.summary_stats(rets_7030b)
summaries.mean()

However, both of these summaries are imperfect, since they aggregate across a wide distribution. In different situations one or the other might make sense, but for most individuals, the range of outcomes are what matters because we observe only one of the different possible scenarios.

## Distribution of Terminal Values and Measuring Risk Budget Efficiency

The basic idea is to measure the distribution of terminal values across all scenarios.

```python
def terminal_values(rets):
    """
    Computes the terminal values from a set of returns supplied as a T x N DataFrame
    Return a Series of length N indexed by the columns of rets
    """
    return (rets+1).prod()

def terminal_stats(rets, floor = 0.8, cap=np.inf, name="Stats"):
    """
    Produce Summary Statistics on the terminal values per invested dollar
    across a range of N scenarios
    rets is a T x N DataFrame of returns, where T is the time-step (we assume rets is sorted by time)
    Returns a 1 column DataFrame of Summary Stats indexed by the stat name 
    """
    terminal_wealth = (rets+1).prod()
    breach = terminal_wealth < floor
    reach = terminal_wealth >= cap
    p_breach = breach.mean() if breach.sum() > 0 else np.nan
    p_reach = breach.mean() if reach.sum() > 0 else np.nan
    e_short = (floor-terminal_wealth[breach]).mean() if breach.sum() > 0 else np.nan
    e_surplus = (cap-terminal_wealth[reach]).mean() if reach.sum() > 0 else np.nan
    sum_stats = pd.DataFrame.from_dict({
        "mean": terminal_wealth.mean(),
        "std" : terminal_wealth.std(),
        "p_breach": p_breach,
        "e_short":e_short,
        "p_reach": p_reach,
        "e_surplus": e_surplus
    }, orient="index", columns=[name])
    return sum_stats

```

In [ ]:
pd.concat([erk.terminal_stats(rets_bonds, name="FI"), 
           erk.terminal_stats(rets_eq, name="Eq"),
           erk.terminal_stats(rets_7030b, name="70/30")],
          axis=1)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 6))
sns.distplot(erk.terminal_values(rets_eq), color="red", label="100% Equities")
sns.distplot(erk.terminal_values(rets_bonds), color="blue", label="100% Bonds")
sns.distplot(erk.terminal_values(rets_7030b), color="orange", label="70/30 Equities/Bonds")
plt.legend();

Note how the higher mean return of the equities portfolio is due in part to the large upside spread in equities.

## Risk Budgeting through Glide Path Allocation

A potential improvement over naive fixed mix is to risk budget by making a simplistic assumption that risk = time.

In other words, if you have more time, increase the risk budget. If you have less time, derisk. This is the thinking behind a Target Date Fund.

Let's write an allocator that defines the weights as a glidepath sthat starts at a starting weight and then gradually decreases the weights to equities over time to an ending weight. For example, you can start at 80% stocks at the start of the 10 year period and then gradually move to 20% stocks by the end of the 10 year period.

Let's see how this does:

```python
def glidepath_allocator(r1, r2, start_glide=1, end_glide=0.0):
    """
    Allocates weights to r1 starting at start_glide and ends at end_glide
    by gradually moving from start_glide to end_glide over time
    """
    n_points = r1.shape[0]
    n_col = r1.shape[1]
    path = pd.Series(data=np.linspace(start_glide, end_glide, num=n_points))
    paths = pd.concat([path]*n_col, axis=1)
    paths.index = r1.index
    paths.columns = r1.columns
    return paths
```


In [ ]:
rets_g8020 = erk.bt_mix(rets_eq, rets_bonds, allocator=erk.glidepath_allocator, start_glide=.8, end_glide=.2)
pd.concat([erk.terminal_stats(rets_bonds, name="FI"), 
           erk.terminal_stats(rets_eq, name="Eq"),
           erk.terminal_stats(rets_7030b, name="70/30"),
           erk.terminal_stats(rets_g8020, name="Glide 80 to 20")],
          axis=1)

In [ ]:
rets_7030z = erk.bt_mix(rets_eq, rets_zc, allocator=erk.fixedmix_allocator, w1=0.7)
plt.figure(figsize=(12, 6))
#sns.distplot(terminal_values(rets_eq), color="red", label="100% Equities")
#sns.distplot(terminal_values(rets_bonds), color="blue", label="100% Bonds")
sns.distplot(erk.terminal_values(rets_7030b), color="orange", label="70/30 Equities/Bonds")
#sns.distplot(terminal_values(rets_g8020), color="green", label="Glide 80 to 20")
sns.distplot(erk.terminal_values(rets_7030z), color="grey", label="70/30 Equities/Zeros")
plt.legend();

### Conclusion

Static or Naive risk budgeting involves allocating between the PSP and GHP using either a simple fixed mix or a blind glidepath. These can reduce the downside risk but come at the cost the expected return, and cannot be used to secure a minimum acceptable level of wealth or liabilities. In the next session, we'll examine dynamic approaches that will address the challenge of meeting a set of future liabilities such as replacement income, or a required level of wealth in the future.


# Monte Carlo simulation of Dynamic Risk Budgeting between PSP and GHP

We've looked at the fundamental problem of how much to allocate in the safe asset vs the performance seeking asset, and we investigated static and glidepath based techniques. Now we'll look at modern dynamic techniques that are inspired by CPPI to ensure that the account value reaches a certain target minimum floor, but also maintains exposure to the upside through a dynamic risk budget

Let's start by building 500 scenarios for interest rates, an duration matched bond portfolio (proxied by a zero coupon bond) and a stock portfolio.

In [ ]:
n_scenarios =  5000
rates, zc_prices = erk.cir(10, n_scenarios=n_scenarios, b=0.03, r_0 = 0.03, sigma=0.02)
price_eq = erk.gbm(n_years=10,n_scenarios=n_scenarios, mu=0.07, sigma=0.15)
rets_eq = price_eq.pct_change().dropna()
rets_zc = zc_prices.pct_change().dropna()
rets_7030b = erk.bt_mix(rets_eq, rets_zc, allocator=erk.fixedmix_allocator, w1=0.7)
pd.concat([erk.terminal_stats(rets_zc, name="ZC", floor=0.75), 
           erk.terminal_stats(rets_eq, name="Eq", floor=0.75),
           erk.terminal_stats(rets_7030b, name="70/30", floor=0.75)],
          axis=1).round(2)

Let's write a new allocator that we'll call a Floor Allocator applies a dynamic risk budget to allocate more to the PSP when there is a cushion, similar to what we did for CPPI

```python
def floor_allocator(psp_r, ghp_r, floor, zc_prices, m=3):
    """
    Allocate between PSP and GHP with the goal to provide exposure to the upside
    of the PSP without going violating the floor.
    Uses a CPPI-style dynamic risk budgeting algorithm by investing a multiple
    of the cushion in the PSP
    Returns a DataFrame with the same shape as the psp/ghp representing the weights in the PSP
    """
    if zc_prices.shape != psp_r.shape:
        raise ValueError("PSP and ZC Prices must have the same shape")
    n_steps, n_scenarios = psp_r.shape
    account_value = np.repeat(1, n_scenarios)
    floor_value = np.repeat(1, n_scenarios)
    w_history = pd.DataFrame(index=psp_r.index, columns=psp_r.columns)
    for step in range(n_steps):
        floor_value = floor*zc_prices.iloc[step] ## PV of Floor assuming today's rates and flat YC
        cushion = (account_value - floor_value)/account_value
        psp_w = (m*cushion).clip(0, 1) # same as applying min and max
        ghp_w = 1-psp_w
        psp_alloc = account_value*psp_w
        ghp_alloc = account_value*ghp_w
        # recompute the new account value at the end of this step
        account_value = psp_alloc*(1+psp_r.iloc[step]) + ghp_alloc*(1+ghp_r.iloc[step])
        w_history.iloc[step] = psp_w
    return w_history
```


In [ ]:
rets_floor75 = erk.bt_mix(rets_eq, rets_zc, allocator=erk.floor_allocator, floor=.75,  zc_prices=zc_prices[1:])
pd.concat([erk.terminal_stats(rets_zc, name="ZC", floor=0.75), 
           erk.terminal_stats(rets_eq, name="Eq", floor=0.75),
           erk.terminal_stats(rets_7030b, name="70/30", floor=0.75),
           erk.terminal_stats(rets_floor75, name="Floor75", floor=0.75),
          ],
          axis=1).round(2)

In [ ]:
rets_floor75m1 = erk.bt_mix(rets_eq, rets_zc, allocator=erk.floor_allocator, zc_prices=zc_prices[1:], floor=.75, m=1)
pd.concat([erk.terminal_stats(rets_zc, name="ZC", floor=0.75), 
           erk.terminal_stats(rets_eq, name="Eq", floor=0.75),
           erk.terminal_stats(rets_7030b, name="70/30", floor=0.75),
           erk.terminal_stats(rets_floor75, name="Floor75", floor=0.75),
           erk.terminal_stats(rets_floor75m1, name="Floor75m1", floor=0.75),
          ],
          axis=1).round(2)

In [ ]:
rets_floor75m5 = erk.bt_mix(rets_eq, rets_zc, allocator=erk.floor_allocator, zc_prices=zc_prices[1:], floor=.75, m=5)
rets_floor75m10 = erk.bt_mix(rets_eq, rets_zc, allocator=erk.floor_allocator, zc_prices=zc_prices[1:], floor=.75, m=10)
pd.concat([erk.terminal_stats(rets_zc, name="ZC", floor=0.75), 
           erk.terminal_stats(rets_eq, name="Eq", floor=0.75),
           erk.terminal_stats(rets_7030b, name="70/30", floor=0.75),
           erk.terminal_stats(rets_floor75, name="Floor75", floor=0.75),
           erk.terminal_stats(rets_floor75m1, name="Floor75m1", floor=0.75),
           erk.terminal_stats(rets_floor75m5, name="Floor75m5", floor=0.75),
           erk.terminal_stats(rets_floor75m10, name="Floor75m10", floor=0.75)
          ],
          axis=1).round(2)

In [ ]:
tv_eq = erk.terminal_values(rets_eq)
tv_zc = erk.terminal_values(rets_zc)
tv_7030b = erk.terminal_values(rets_7030b)
tv_floor75 = erk.terminal_values(rets_floor75)
tv_floor75m1 = erk.terminal_values(rets_floor75m1)

plt.figure(figsize=(12, 6))
sns.distplot(tv_eq, color="red", label="100% Equities", bins=100)
plt.axvline(tv_eq.mean(), ls="--", color="red")
#sns.distplot(tv_zc, color="blue", label="100% Immunized Bonds")
#plt.axvline(tv_zc.mean(), ls="--", color="blue")
sns.distplot(tv_7030b, color="orange", label="70/30 Equities/Bonds", bins=100)
plt.axvline(tv_7030b.mean(), ls="--", color="orange")
sns.distplot(tv_floor75, color="green", label="Floor at 75%", bins=100)
plt.axvline(tv_floor75.mean(), ls="--", color="green")

plt.legend();

## Dynamic Risk Budgeting with Drawdown instead of a fixed floor

```python
def drawdown_allocator(psp_r, ghp_r, maxdd, m=3):
    """
    Allocate between PSP and GHP with the goal to provide exposure to the upside
    of the PSP without going violating the floor.
    Uses a CPPI-style dynamic risk budgeting algorithm by investing a multiple
    of the cushion in the PSP
    Returns a DataFrame with the same shape as the psp/ghp representing the weights in the PSP
    """
    n_steps, n_scenarios = psp_r.shape
    account_value = np.repeat(1, n_scenarios)
    floor_value = np.repeat(1, n_scenarios)
    ### For MaxDD
    peak_value = np.repeat(1, n_scenarios)
    w_history = pd.DataFrame(index=psp_r.index, columns=psp_r.columns)
    for step in range(n_steps):
        ### For MaxDD
        floor_value = (1-maxdd)*peak_value ### Floor is based on Prev Peak
        cushion = (account_value - floor_value)/account_value
        psp_w = (m*cushion).clip(0, 1) # same as applying min and max
        ghp_w = 1-psp_w
        psp_alloc = account_value*psp_w
        ghp_alloc = account_value*ghp_w
        # recompute the new account value at the end of this step
        account_value = psp_alloc*(1+psp_r.iloc[step]) + ghp_alloc*(1+ghp_r.iloc[step])
        ### For MaxDD
        peak_value = np.maximum(peak_value, account_value) ### For MaxDD
        w_history.iloc[step] = psp_w
    return w_history
```

In [ ]:
cashrate = 0.02
monthly_cashreturn = (1+cashrate)**(1/12) - 1
rets_cash = pd.DataFrame(data= monthly_cashreturn, index=rets_eq.index, columns=rets_eq.columns)
rets_maxdd25 = erk.bt_mix(rets_eq, rets_cash, allocator=erk.drawdown_allocator, maxdd=.25)
tv_maxdd25 = erk.terminal_values(rets_maxdd25)
pd.concat([erk.terminal_stats(rets_zc, name="ZC", floor=0.75), 
           erk.terminal_stats(rets_eq, name="Eq", floor=0.75),
           erk.terminal_stats(rets_7030b, name="70/30", floor=0.75),
           erk.terminal_stats(rets_floor75, name="Floor75", floor=0.75),
           erk.terminal_stats(rets_floor75m1, name="Floor75m1", floor=0.75),
           erk.terminal_stats(rets_floor75m5, name="Floor75m5", floor=0.75),
           erk.terminal_stats(rets_floor75m10, name="Floor75m10", floor=0.75),
           erk.terminal_stats(rets_maxdd25, name="MaxDD25", floor=0.75)           
          ],
          axis=1).round(2)

In [ ]:
erk.summary_stats(rets_maxdd25)["Max Drawdown"].min()

In [ ]:
# Plot
plt.figure(figsize=(12, 6))
sns.distplot(tv_eq, color="red", label="100% Equities", bins=100)
plt.axvline(tv_eq.mean(), ls="--", color="red")
#sns.distplot(tv_zc, color="blue", label="100% Immunized Bonds")
#plt.axvline(tv_zc.mean(), ls="--", color="blue")
sns.distplot(tv_7030b, color="orange", label="70/30 Equities/Bonds", bins=100)
plt.axvline(tv_7030b.mean(), ls="--", color="orange")
sns.distplot(tv_floor75, color="green", label="Floor at 75%", bins=100)
plt.axvline(tv_floor75.mean(), ls="--", color="green")
sns.distplot(tv_maxdd25, color="yellow", label="MaxDD = 25%", bins=100)
plt.axvline(tv_maxdd25.mean(), ls="--", color="yellow")
plt.legend();

## Backtesting Dynamic Strategies with Historical Data

Try and work with real historic data such as a the Industry Portfolios or the Total Market Index we constructed and run back tests with different parameter values. For instance:

In [ ]:
rets_tmi =  erk.get_total_market_index_returns()["1990":]
dd_tmi = erk.drawdown(rets_tmi)
ax = dd_tmi["Wealth"].plot(figsize=(12,6), ls="-", color="goldenrod")
dd_tmi["Previous Peak"].plot(ax=ax, ls=":", color="goldenrod")

In [ ]:
cashrate = 0.03
monthly_cashreturn = (1+cashrate)**(1/12) - 1
rets_cash = pd.DataFrame(data= monthly_cashreturn, index=rets_tmi.index, columns=[0]) # 1 column dataframe
rets_maxdd25 = erk.bt_mix(pd.DataFrame(rets_tmi), rets_cash, allocator=erk.drawdown_allocator, maxdd=.25, m=5)
dd_25 = erk.drawdown(rets_maxdd25[0])

ax = dd_tmi["Wealth"].plot(figsize=(12,6), ls="-", color="goldenrod", label="Market", legend=True)
dd_tmi["Previous Peak"].plot(ax=ax, ls=":", color="goldenrod")
dd_25["Wealth"].plot(ax=ax, label="MaxDD 25%", color="cornflowerblue", legend=True)
dd_25["Previous Peak"].plot(ax=ax, color="cornflowerblue", ls=":")

In [ ]:
erk.summary_stats(pd.concat([rets_tmi.rename("Market"), rets_maxdd25[0].rename("MaxDD")], axis=1))


---

# Chapter 4 (Lab): Asset-Liability Management

---


## Asset-Liability Management

In [ ]:
import pandas as pd
import numpy as np
import edhc_risk_kit as erk
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline
%load_ext autoreload
%autoreload 2

Asset Liability Management (ALM) can be defined as a mechanism to address the risk faced by a bank due to a mismatch between assets and liabilities either due to liquidity or changes in interest rates. Liquidity is an institution’s ability to meet its liabilities either by borrowing or converting assets. Apart from liquidity, a bank may also have a mismatch due to changes in interest rates as banks typically tend to borrow short term (fixed or floating) and lend long term (fixed or floating).

# Present Value of Liabilities and Funding Ratio

We will examine how to discount future liabilities to compute the present value of future liabilities, and measure the funding ratio. The funding ratio is the ratio of the current value of assets to the present value of the liabilities. In order to compute the present value, we need to discount the amount of the liability based on the relevant interest rate derived from the yield curve. For simplicity, we'll assume that the yield curve is flat, and so the interest rate is the same for all horizons.

The present value of a set of liabilities $L$ where each liability $L_i$ is due at time $t_i$ is give by:

$ PV(L) = \sum_{i=1}^{k} B(t_i) L_i$

where $B(t_i)$ is the price of a pure discount bond that pays 1 dollar at time $t_i$

If we assume the yield curve is flat and the annual rate of interest is $r$ then $B(t)$ is given by

$$B(t) = \frac{1}{(1+r)^t}$$

Assume that you have 4 liabilities, of 1, 1.5, 2, and 2.5M dollars. Assume the first of these are 3 years away and the subsequent ones are spaced out 6 months apart, i.e. at time 3, 3.5, 4 and 4.5 years from now. Let's compute the present value of the liabilities based on an interest rate of 3% per year.

In an individual investment context, you can think oif liabilities as Goals, such as saving for Life Events such as a down payment for a house, college expenses for your children, or retirement income. In each of these cases, we have a requirement of a cash flow at some point in the future ... anytime you have a future cash requirement, you can think of it as a liability.

In [ ]:
# Example calculating discounted value
erk.discount_simple(10, .03)

In [ ]:
# Example calculating present value
liabilities = pd.Series(data=[1, 1.5, 2, 2.5], index=[3, 3.5, 4, 4.5])
erk.pv_simple(liabilities, 0.03)

We can now compute the funding ratio, based on current asset values:

In [ ]:
# Example calculating the funding ration
erk.funding_ratio_simple(5, liabilities, 0.03)

Thus, in order to be able to meet the future liabilities as specified in `liabilities` at the indicated future times, the present funding ratio should be 80.2% (6.23 million in current assets in order to pay 7 million at years 3, 3.5, 4, and 4.5).

As the widget below shows, even if your assets do not go down in value, cash can be a risky asset if you think about the funding ratio rather than the asset value. Even though cash is a "safe asset" in the sense that the asset value does not go down, cash can be a very risky asset because the value of the liabilities goes up when interest rates go down. Therefore, if you think about your savings in terms of funding ratio (i.e. how much money do you have compared to what you need) then cash is a risky asset and can result in a decline in your funding ratio.

In [ ]:
def show_funding_ratio(assets, r):
    fr = erk.funding_ratio_simple(assets, liabilities, r)
    print(f'{fr*100:.2f}%')
    
controls = widgets.interactive(show_funding_ratio,
                                   assets=widgets.IntSlider(min=1, max=10, step=1, value=5),
                                   r=(0, .20, .01)
)
display(controls)

# Simulating Interest Rate Changes and Liability Hedging
### Implementing the Cox Ingersoll Ross Model for interest rates

We saw above that when you look at your financial condition in terms of funding ratio, rather than asset value, even cash can be a risky asset.

We're going to simulate changes in interest rates using the Cox-Ingersoll-Ross model, usually just called the CIR model. It is very similar to what we have already done for generating random stock prices. Instead of a Geometric Brownian Motion, the mean reversion model features parameters to control the speed of mean reversion as well as the long term average rate.

$$ dr_{t}=a(b-r_{t})\,dt+\sigma {\sqrt  {r_{t}}}\,dW_{t} $$

where $a$ is the speed of mean reversion and $b$ is the long term mean rate and $\sigma$ is the volatility parameter.

This can be read as describing the change in interest at any point of time as a multiple of the distance of the current rate from it's long term mean plus a random value that depends on the volatility parameter.

Thus, we can start at some rate $r_0$ and generate a sequence of rates over time using the change in rate defined by the stochastic differential equation above. If we eliminate the ${\sqrt  {r_{t}}}$ term we get the simpler Vasicek model, but the CIR model eliminates the possibility of negative interest rates by shutting down the volatility term as interest rates approach zero. 

### Short Rate vs Annualized Rate

One final piece of math we'll need is that the interest rate being described here is the "instantaneous" or "short rate" ... we will often want to think about interest rates as annualized rates, which means that the instantaneous rate needs to be compounded into an annual rate as follows:

$$ 1 + r_{annual} = e^{r_{inst}} $$

The function `inst_to_ann` converts an instantaneous (or short) rate to an annulaized rate while the function `ann_to_inst` does the opposite. The function `cir` implements the CIR model (see `edhc_risk_kit.py`).

In [ ]:
rates, prices = erk.cir(n_scenarios=10)
rates.plot(legend=False, figsize=(12,5))
plt.xlabel('Periods')
plt.ylabel('Interest Rate');

In [ ]:
def show_cir(r_0=0.03, a=0.5, b=0.03, sigma=0.05, n_scenarios=5):
    rates, prices = erk.cir(r_0=r_0, a=a, b=b, sigma=sigma, n_scenarios=n_scenarios)
    rates.plot(legend=False, figsize=(12,5))
    
import ipywidgets as widgets
from IPython.display import display
    
controls = widgets.interactive(show_cir,
                              r_0 = (0, .15, .01),
                              a = (0, 1, .1),
                               b = (0, .15, .01),
                               sigma= (0, .1, .01),
                               n_scenarios = (1, 100))
display(controls)

## Generating the random price evolution of a Zero-Coupon Bond

The model can also be used to generate the movement of bond prices for a zero coupon bond that are implied by the generated interest rate, using the following equations:

$$P(t,T)=A(t,T)e^{-B(t,T)r_{t}}\!$$

where

$$ A(t,T)=\left({\frac  {2h e^{(a+h)\tau/2}}{2h+(a+h)(e^{\tau h}-1)}}\right)^{{2ab/\sigma ^{2}}} $$
and
$$ B(t,T)={\frac  {2(e^{\tau h}-1)}{2h+(a+h)(e^{\tau h}-1)}} $$
and
$$ h={\sqrt  {a^{2}+2\sigma ^{2}}} $$
and
$$ \tau = T-t $$


In [ ]:
def show_cir_prices(r_0=0.03, a=0.5, b=0.03, sigma=0.05, n_scenarios=5):
    rates, prices = erk.cir(r_0=r_0, a=a, b=b, sigma=sigma, n_scenarios=n_scenarios)
    prices.plot(legend=False, figsize=(12,5))

controls = widgets.interactive(show_cir_prices,
                              r_0 = (0, .15, .01),
                              a = (0, 1, .1),
                               b = (0, .15, .01),
                               sigma= (0, .1, .01),
                               n_scenarios = (1, 100))
display(controls)

Assume our liabilities are to pay one dollar in 10 years time. Clearly, the liability is perfectly matched by the price of a zero coupon bond that matures in 10 years. Therefore, this is perfectly matched by buying a zero coupon bond. 

In [ ]:
# my starting asset value today aka cash in hand
a_0 = 0.75
# simulate the next 10 years
rates, bond_prices = erk.cir(r_0=0.03, b=0.03, n_scenarios=10)
# liabilities are going to change over time to match the payoff of the ZCB
liabilities = bond_prices.copy()
# today's zero coupon bond price is the present value of 1 dollar in 10 years
zc_0 = erk.pv(pd.Series(data=[1], index=[10]), 0.03)
# I can put my money in a zero coupon bond at today's price
n_bonds = float(a_0/zc_0)
# and my asset value will grow based on the bond prices
av_zc_bonds = n_bonds*bond_prices
# how about if I invest in cash? It will grow at some rate of interest
av_cash = a_0*(rates/12+1).cumprod()

In [ ]:
av_cash.plot(title='Asset Values using Cash (10 scenarios)', legend=False, figsize=(12,5));

In [ ]:
av_zc_bonds.plot(title='Asset Values using ZC Bonds (10 scenarios)', legend=False, figsize=(12,5))

In [ ]:
av_cash.pct_change().plot(title='Returns using Cash (10 scenarios)', legend=False, figsize=(12,5));

In [ ]:
av_zc_bonds.pct_change().plot(title='Returns using ZC Bonds (10 scenarios)', legend=False, figsize=(12,5));

In [ ]:
(av_cash/liabilities).pct_change().plot(title='Returns of Funding Ratio with Cash (10 scenarios)', legend=False, figsize=(12,5));

In [ ]:
(av_zc_bonds/liabilities).pct_change().plot(title='Returns of Funding Ratio with ZC Bonds (10 scenarios)', legend=False, figsize=(12,5));

In [ ]:
av_zc_bonds.plot(title='Asset Values using ZC Bonds (10 scenarios)', legend=False, figsize=(12,5));

### Terminal Funding Ratios

We can look at this in terms of Terminal Funding Ratios. Let's repeat this with a large number of simulations:

In [ ]:
# my starting asset value today aka cash in hand
a_0 = 0.75
# simulate the next 10 years
rates, bond_prices = erk.cir(r_0=0.03, b=0.03, n_scenarios=10000)
# liabilities are going to change over time to match the payoff of the ZCB
liabilities = bond_prices
# today's zero coupon bond price is the present value of 1 dollar in 10 years
zc_0 = erk.pv(pd.Series(data=[1], index=[10]), 0.03)
# I can put my money in a zero coupon bond at today's price
n_bonds = a_0/zc_0
# and my asset value will grow based on the bond prices
av_zc_bonds = n_bonds*bond_prices
# how about if I invest in cash? It will grow at some rate of interest over time
av_cash = a_0*(rates/12+1).cumprod()

In [ ]:
tfr_cash = av_cash.iloc[-1]/liabilities.iloc[-1]
tfr_zc_bonds = av_zc_bonds.iloc[-1]/liabilities.iloc[-1]
ax = tfr_cash.plot.hist(label="Cash", figsize=(12,5), bins=100, legend=True)
tfr_zc_bonds.plot.hist(ax=ax,label="ZC Bonds", bins=100, secondary_y=True, legend=True);

### Conclusion

As you can see, hedging a future liability with cash or even cash-like instruments can be very risky. Although Zero Coupon Bonds are nominally risky and can see a great deal of price fluctuation, they vary in price in tandem with the liabilities, resulting in a much better hedge.

Of course, this makes two convenient assumptions that are often not true in practice. First, it assumes that zero coupon bonds with the perfect maturities are always available, and second, it assumes that you have the intial assets to be able to afford the hedge.

We shall try and address these later. We address the first by holding a portfolio of non zero coupon bonds that will behave like a zero coupon bond in the sense that it can be liquidated at a particular maturity for an amount that is immunized against changes in interest rates. We'll address the second issue by introducing a second building block to deliver the higher performance to try and close the gap between your current assets and the upside target.

# GHP Construction with Duration matching

As seen before, we can easily match a liability if we had a zero coupon bond that had the exact maturity as the liability, in which case, assuming we had enough funds available, we could do perfect cash flow matching. Next, we'll use a portfolio of standard, non-coupon bonds to try and achieve the same behaviour as a zero coupon bond.

In order to do this, we'll have to find a way to price a bond, but as it turns out, we've already written the code to do this. Recall that we implemented the code to compute the present value of a sequence of liability cash flows. Well, a bond is nothing more than a series of cash flows that the issuer is offering to make to you, and therefore, the price of a bond is nothing more than the present value of the future cash flows. To be more precise, we should use discount rates matching the payment horizons using the yield curve. For simplicity, we'll assume that the yield curve is flat, and the rate of interest is the same regardless of whether it is 1 year, 5 year, 10 year or 30 year horizon. We'll also assume that bonds pay a monthly coupon thereby avoiding some of the complexities of intra-coupon payment. 

In [ ]:
# Example for a bond which matures in 20 years, a face value of 1000, a coupon rate of 5%
# two coupons per year and a discount rate of 4%
print(erk.bond_price(20, 1000, 0.05, 2, .04))

## Bond prices change with interest rates

Consider the following chart that shows the change in value of a bond for interest rates from 2% to 10%. The bond in question is a 10 year bond that pays a 5% coupon semiannually and the face value is $1000.

In [ ]:
rates = np.linspace(.01, .10, num=20)
prices = [erk.bond_price(10, 1000, 0.05, 2, rate) for rate in rates]
pd.DataFrame(data=prices, index=rates).plot(title="Change in 10Y Bond Prices with Interest Rates", legend=False);

Assume that we have a liability of \$1000 in 10 years. Clearly, we cannot use this as a hedge, since we would be severely underfunded if interest rates rise, and over funded if they fall. If we had a Zero Coupon bond, we could purchase that, but Zero Coupon bonds of specific maturities arent always available.

What is available are various bonds with maturities from 5 to 30 years. How do we build a portfolio that has the same sensitivity to interest rates as our liabilities do? If they both respond the same way to changes in interest rates, we can effectively obtain immunity to interest rate changes.

One way to do this is through _Duration Matching_. The duration of a bond measures its senstivity to interest rate changes. Therefore, we match the duration of the liabilities to the duration of the bond portfolio. As it happens, the duration of a bond which is computed as the weighted average time of the cash flows, weighted by the cash flows.

Thus the duration of a Zero Coupon bond is just the maturity, since there is a single cash flow at maturity.

_______________________________________

The Macaulay Duration which is the weighted average number of periods you have to be invested in the bond to get the present value of the bond back. In this example, we'll need to wait about 5.58/2 = 2.79 years on average to get back the present value of the bond. You can also compute something called the _Modified Duration_ which gives you a more direct measure of how much the bond price will move for a small change in interest rates (i.e. the first derivative of the bond price with respect to interest rates). However, we won't need to compute that since the Macaulay Duration is itself a measure of sensitivity and all we need to do is to match that to our liability.

In [ ]:
erk.macaulay_duration(erk.bond_cash_flows(3, 1000, .06, 2), 0.06/2)

# Duration Matching

Assume we have a liability of 100K in 10 years time and another of 100K in 12 years time. Assume interest rates are 4%. What is the duration of the liabilities?

In [ ]:
liabilities = pd.Series(data = [100000, 100000], index=[10, 12])
erk.macaulay_duration(liabilities, .04)

Now assume we have two types of bonds available. We have a 10 year bond and a 15 year bond. Each of them pays a 5% coupon once a year and has a face value of \\$1000. What are the durations of these bonds?

In [ ]:
md_10 = erk.macaulay_duration(erk.bond_cash_flows(10, 1000, .05, 1), .04)
print(md_10)
md_20 = erk.macaulay_duration(erk.bond_cash_flows(20, 1000, .05, 1), .04)
print(md_20)

Therefore, we need to hold a portfolio of these two bonds that has a combined target duration that matches the duration of the liability, which is given by the following expression, where $w_s$ is the weight in the short duration bond whcih has duration $d_s$ and the duration of the longer bond is $d_l$. We designate the targeted duration as $d_t$.

In our case, the fraction in the short duration asset $w_s$ should be such that:

$$ w_s \times 8.19 + (1-w_s) \times 13.54 = 10.96 $$

more generally:

$$ w_s \times d_s + (1-w_s) \times d_l = d_t $$

rearranging gives:

$$ w_s = \frac{d_l -d_t}{d_l - d_s} $$

The function `match_duration` in the `edhc_risk_kit` will do this for us.

In [ ]:
# Example
short_bond = erk.bond_cash_flows(10, 1000, .05, 1)
long_bond = erk.bond_cash_flows(20, 1000, .05, 1)
w_s = erk.match_durations(liabilities, short_bond, long_bond, 0.04)
w_s # the weight for the short bond

In [ ]:
p_short = erk.bond_price(10, 1000, .05, 1, 0.04)
p_long = erk.bond_price(20, 1000, .05, 1, 0.04)
a_0 = 130000
dm_assets=pd.concat([a_0*w_s*short_bond/float(p_short),a_0*(1-w_s)*long_bond/float(p_long)])
erk.macaulay_duration(dm_assets, 0.04)

In [ ]:
cfr = erk.funding_ratio(dm_assets, liabilities, 0.04)
cfr

In [ ]:
lb_assets = a_0*long_bond/float(p_long)
sb_assets = a_0*short_bond/float(p_short)

In [ ]:
rates = np.linspace(0, .1, 20)
fr_change = pd.DataFrame({
    "Long Bond":[erk.funding_ratio(lb_assets, liabilities, r) for r in rates],
    "Short Bond":[erk.funding_ratio(sb_assets, liabilities, r) for r in rates],
    "Duration Matched Bonds":[erk.funding_ratio(dm_assets, liabilities, r) for r in rates]
}, index=rates)
fr_change.plot(title='Change in Funding Ratio', figsize=(10,6));

## Conclusion

We've seen that if you have the assets on hand, you can secure future liabilities by matching the duration of your liabilities with the durations of your assets to immunize yourself against level shifts in the yield curve. In practice, you might also want to protect against changes in the slope and curvature of the yield curve by using convexity hedging, but duration hedging is a simple but essential step towards ensuring that you will have the money to meet your liabilities when interest rates fluctuate.


# Monte Carlo Simulation of Prices of Coupon-Bearing Bond using CIR

As we have seen in the previous sessions, if we are fully funded, the best way to ensure that you will remain fully funded and be able to meet your liabilities is to build a portfolio of immunized bonds that are designed to match the future liability cash flows.

In practice, few of us, both in a Pension Fund context and even more so in a Personal Investment context, already have the assets to fund all future liabilities. Other than in cases where 100% funding ratios are mandated by law, the far more common situation is that we typically do not have the assets we need to fully fund our future liabilities.

That is where we need to allocate the wealth that we do have between the Performance Seekeng Component, and the Goal Hedging Component that we have already seen.

Therefore the next steps for us are to allocate our capital across two assets: the PSP and the GHP. The PSP, or Performance Seeking Portfolio, is where we consciously take some risk in the hope that the extra risk is well rewarded - in other words, a high Sharpe Ratio portfolio. The GHP or Goal Hedging Portfolio is the immnized bond portfolio that designed to protect us in case the worst case happens and we do not get the returns that we hope for from the PSP.

This of course raises the question of how much to allocate to the PSP and how much to the GHP. Clearly, allocating more to the PSP entails a higher expected return but a higher degree of risk. On the other hand, allocating too little to the PSP means less risk, but lower expected return. Thus, the decision of how much to allocate to the PSP vs the GHP is essentially a risk budgeting exercise.

We'll start out by developing the framework and in the next few lab sessions, examine different risk budgeting options.

First, we'll need to make a choice of the PSP. For now, we'll start with something fairly simple, which is the Geometric Brownian Motion based random stock portfolios. Much of the second MOOC in this series is focused on how to build good PSPs in practice.

However, we'll also need to pick a GHP. We can use Zero Coupon Bonds driven by random changes in interest rates using the CIR model. Since these are Zero Coupon, they are already immunized against interest rates if they are held to maturity. In practice, we've seen that even if we arent able to buy Zero Coupon bonds, we can construct bond portfolios that are designed to be insensitive to interest rates through duration matching.

In this section, we'll develop the code necessary to simulate bond prices of coupon-bearing bonds and see how their prices change as interest rates fluctuate. We'll limit ourselves to Treasury Bonds and ignore credit risk (i.e. the risk of default on the Bond).

_______________________________

Note that when interest rates rise, it is a fallacy that holding an allocation to bonds will give you the benefit of that increase. In fact, the opposite happens since the price of the bond will fall, and as a result your account sill suffer a capital loss.

Let's examine what happens to your wealth when you use a portfolio of stocks and bonds over a 5 year period.

For simplicity, we'll assume you are holding a bond that has a maturity of 5 years and for simplicity with avoiding intra-coupon caclulations, let's assume it pays a 5% coupon and the coupon is paid out each month and interest rates change from 3% to 3.2%


In [ ]:
erk.bond_price(5,100,.05,12,.03)

In [ ]:
erk.bond_price(5,100,.05,12,.032)

We'll generate interest rates using the CIR model:

In [ ]:
rates, zc_prices = erk.cir(10, 500, b=0.03, r_0 = 0.03)

When we start out, all rates are at 3% and so the prices are all the same:

In [ ]:
erk.bond_price(5,100,.05,12, rates.iloc[0][[1,2,3]])

In [ ]:
rates[[1,2,3]].head()

At t=0 interest rates are the same across all scenarios. However at the first step, we see that (i) the maturity decreases and (ii) interest rates change, and so the bond prices diverge:

In [ ]:
erk.bond_price(5,100,.05,12, rates.iloc[1][[1,2,3]])

### Simulating Prices of a Coupon-bearing Bond

From the ability to compute bond prices from rates, we can stitch together the prices of a bond over time:

In [ ]:
erk.bond_price(10, 100, .05, 12, rates[[1,2,3,4]]).plot(legend=False, figsize=(12,6));

In [ ]:
prices = erk.bond_price(10, 100, .05, 12, rates[[1,2,3,4]])
prices.head()

Because the interest rate at the time we bought the bond was less than the coupon rate, we bought it at a premium, taking a capital loss when we sold it at the end. We need to compute the Total Return of a bond, which is the price return PLUS the dividend with the function `bond_total_return`.

In [ ]:
p = erk.bond_price(10, 100, .05, 12, rates[[1,2,3,4]])
btr = erk.bond_total_return(p, 100, .05, 12)
erk.annualize_rets(btr, 12)

Which gives us the approximately 3% return we expected, because that was the prevailing rate when we bought it. We also assumed that we reinvested the coupon in the bond and that is why we did not get the exact same return in each case, since we would observe prices based on the then-prevalent interest rates.

## Putting it all together: Monte Carlo Simulation of Asset Allocation

Now that we have a way to generate prices from which to derive returns, we can experiment with allocating across the different Asset Classes.

Let's start by examining the performance of a 70-30 allocation to Stocks and Bonds. Assume Stocks return an average of 7% per year with a 15% vol and use the CIR model to generate bond prices for a 10 year and 30 year bond that pays a 5% coupon. For simplicity, assume the coupon is paid monthly to avoid having to deal with partial coupons. Assume the Bond Portfolio consists of 60% in the 10 year bond and 40% in the 30 year bond.

In [ ]:
price_10 = erk.bond_price(10, 100, .05, 12, rates)
price_30 = erk.bond_price(30, 100, .05, 12, rates)
rets_30 = erk.bond_total_return(price_30, 100, .05, 12)
rets_10 = erk.bond_total_return(price_10, 100, .05, 12)
rets_bonds = .6*rets_10 + .4*rets_30 # assume monthly rebalancing
mean_rets_bonds = rets_bonds.mean(axis='columns')
erk.summary_stats(pd.DataFrame(mean_rets_bonds))

Similarly, for a mix of equities (70%) and bonds (30%).

In [ ]:
price_eq = erk.gbm(n_years=10,n_scenarios=500,mu=0.07, sigma=0.15)
rets_eq = price_eq.pct_change().dropna()
rets = .70*rets_eq + 0.3*rets_bonds
rets_mean = rets.mean(axis='columns')
erk.summary_stats(pd.DataFrame(rets_mean))

# Naive Risk Budgeting Strategies between PSP and GHP

## The Simplest Allocator - Fixed Mix

The allocator's job is to come up with a time series of weights, so let's create the simplest possible allocator - one that puts a fixed fraction in the first portfolio and the remaining in the second.

Let's run the experiment we ran before ... a bond portfolio of 60% in the 10 year bond and 40% in the 30 year bond to generate a fixed mix bond portfolio.

In [ ]:
rates, zc_prices = erk.cir(10, 500, b=0.03, r_0 = 0.03)
price_10 = erk.bond_price(10, 100, .05, 12, rates)
price_30 = erk.bond_price(30, 100, .05, 12, rates)
rets_30 = erk.bond_total_return(price_30, 100, .05, 12)
rets_10 = erk.bond_total_return(price_10, 100, .05, 12)
rets_bonds = erk.bt_mix(rets_10, rets_30, allocator=erk.fixedmix_allocator, w1=.6)
mean_rets_bonds = rets_bonds.mean(axis='columns')
erk.summary_stats(pd.DataFrame(mean_rets_bonds))

TBD 128.ipynb

**Questions 1-3**  

- B1 is a 15 Year Bond with a Face Value of 1000 that pays a 5% coupon semi-annually (2 times a year)  
- B2 is a 5 Year Bond with a Face value of 1000 that pays a 6% coupon quarterly (4 times a year)
- B3 is a 10 Year Zero-Coupon Bond with a Face Value of 1000

Which of the three bonds is the most expensive? Which of the three bonds is the least expensive?

In [ ]:
B1 = erk.bond_price(15, 1000, 0.05, 2, 0.05)
B2 = erk.bond_price(5, 1000, 0.06, 4, 0.05)
print('Price of B1:', float(B1))
print('Price of B2:', float(B2))
print('Price of B3', 1000/(1.05)**10)

**Questions 4-6**  
Which of the three bonds has the highest (Macaulay) Duration?

In [ ]:
D1 = erk.macaulay_duration(erk.bond_cash_flows(15, 1000, 0.05, 2), 0.05/2) / 2
D2 = erk.macaulay_duration(erk.bond_cash_flows(5, 1000, 0.06, 4), 0.05/4) / 4
#D3 = erk.macaulay_duration(erk.bond_cash_flows(10, 1000, 0.00), 0.05)
print('Duration of B1:', D1)
print('Duration of B2:', D2)
print('Duration of B3:', 9.52)

**Question 7**  
Assume a sequence of 3 liabilities of 100,000, 200,000 and 300,000 that are 3, 5 and 10 years away, respectively. What is the Duration of the liabilities?

In [ ]:
liabilities7 = pd.Series(data = [100000, 200000, 300000], index=[3, 5, 10])
erk.macaulay_duration(liabilities7, .05)

**Question 8**  
Assuming the same set of liabilities as the previous question (i.e. a sequence of 3 liabilities of 100,000, 200,000 and $300,000 that are 3, 5 and 10 years away, respectively) build a Duration Matched Portfolio of B1 and B2 to match these liabilities. What is the weight of B2 in the portfolio? (Hint: the code we developed in class erk.match_durations() assumes that all the bonds have the same number of coupons per year. This is not the case here, so you will either need to enhance the code or compute the weight directly e.g. by entering the steps in a Jupyter Notebook Cell or at the Python Command Line)

**Question 9**  
Assume you can use any of the bonds B1, B2 and B3 to build a duration matched bond portfolio matched to the liabilities. Which combination of 2 bonds can you NOT use to build a duration matched bond portfolio?

**Question 10**  
Assuming the same liabilities as the previous questions (i.e. a sequence of 3 liabilities of 100,000, 200,000 and 300,000 that are 3, 5 and 10 years away, respectively), build a Duration Matched Portfolio of B2 and B3 to match the liabilities.  
What is the weight of B2 in this portfolio?

In [ ]:
# Example
short_bond = erk.bond_cash_flows(5, 1000, .05, 4)
long_bond = erk.bond_cash_flows(10, 1000, .05, 1)
w_s = erk.match_durations(liabilities7, short_bond, long_bond, 0.05)
w_s # the weight for the short bond